# E-CUP 2026 — Phase 3: Qwen3.5-4B QLoRA

Два независимых адаптера одновременно:

- **T4:0 → БАД**
- **T4:1 → Легковоспламеняющиеся**

Обучаем модель выдавать **напрямую `label` 0/1 из `data.csv`**. Никакого `бан ↔ label` внутри обучения нет.

После обучения адаптеры сохраняются отдельно и архивируются в ZIP.

In [1]:
# 0. Dependencies
!pip install -q -U "transformers==5.14.0" peft accelerate bitsandbytes safetensors huggingface_hub
!pip cache purge >/dev/null 2>&1 || true
!pip install -q causal-conv1d flash-linear-attention || true
print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 91.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 44.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 101.5 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
  Installing build dependencies ... done
  Getting requirements to bui

In [2]:
# 1. Config
from pathlib import Path
import os, gc, json, math, random, shutil, subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

import torch
from huggingface_hub import snapshot_download
from IPython.display import FileLink, display

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
VAL_N = 600
MODEL_ID = "Qwen/Qwen3.5-4B"

CONTACT_SHEET_SIZE = 576
MAX_IMAGES = 5
JPEG_QUALITY = 88

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LR = 1e-4
EPOCHS = 1
GRAD_ACCUM = 8
MAX_DESCRIPTION_CHARS = 2200
BALANCED_SAMPLING = True

ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working/ecup_phase3")
HF_ROOT = Path("/kaggle/working/hf_cache")
SHEETS_DIR = WORK/"contact_sheets"
ADAPTERS_DIR = WORK/"adapters"

for p in [WORK, HF_ROOT, SHEETS_DIR, ADAPTERS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_ROOT)
os.environ["HF_HUB_CACHE"] = str(HF_ROOT/"hub")
os.environ["HF_XET_CACHE"] = str(HF_ROOT/"xet")

assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 2, "Выбери Kaggle T4 x2"

for i in range(2):
    p = torch.cuda.get_device_properties(i)
    print(i, torch.cuda.get_device_name(i), round(p.total_memory/2**30,2), "GB")

def disk_status(title):
    total, used, free = shutil.disk_usage("/kaggle/working")
    print(f"{title}: used={used/2**30:.2f} GB free={free/2**30:.2f} GB")

disk_status("Start")

0 Tesla T4 14.56 GB
1 Tesla T4 14.56 GB
Start: used=0.00 GB free=19.50 GB


In [3]:
# 2. Locate dataset / images
def find_data_csv(root):
    files = list(root.rglob("data.csv"))
    if not files:
        raise FileNotFoundError("data.csv not found")
    return files[0]

def find_images_root(root):
    candidates = []
    for p in root.rglob("images"):
        if not p.is_dir():
            continue
        dirs = [x for x in list(p.iterdir())[:100] if x.is_dir()]
        if dirs:
            score = sum(x.name.isdigit() for x in dirs) / len(dirs)
            if score >= 0.7:
                candidates.append((score,p))
    if not candidates:
        raise FileNotFoundError("images/<id> not found")
    return max(candidates, key=lambda x:x[0])[1]

DATA_CSV = find_data_csv(ROOT)
IMAGES_ROOT = find_images_root(ROOT)
df = pd.read_csv(DATA_CSV)

IMG_EXTS = {".jpg",".jpeg",".png",".webp"}

def id_str(x):
    if isinstance(x,(float,np.floating)) and float(x).is_integer():
        return str(int(x))
    return str(x)

def sort_key(p):
    try: return (0,int(p.stem))
    except: return (1,p.name)

def get_images(pid):
    folder = IMAGES_ROOT/id_str(pid)
    if not folder.exists():
        return []
    return sorted(
        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS],
        key=sort_key
    )[:MAX_IMAGES]

df["image_paths"] = [get_images(x) for x in tqdm(df["id"], desc="Link images")]
df["n_images"] = df["image_paths"].map(len)

print("Rows:", len(df))
display(pd.crosstab(df["category"], df["label"], margins=True))

Link images:   0%|          | 0/12971 [00:00<?, ?it/s]

Rows: 12971


label,0,1,All
category,,,
БАД,1905,5564,7469
Легковоспламеняющиеся,5304,198,5502
All,7209,5762,12971


In [4]:
# 3. Fixed validation (reuse Phase 2 if provided)
validation_files = list(ROOT.rglob("validation_ids.csv"))

if validation_files:
    fixed = pd.read_csv(validation_files[0])
    ids = set(fixed["id"].astype(str))
    mask = df["id"].astype(str).isin(ids)
    val_df = df[mask].copy().reset_index(drop=True)
    train_df = df[~mask].copy().reset_index(drop=True)
    assert len(val_df) == len(fixed)
    print("Reused:", validation_files[0])
else:
    strata = df["category"].astype(str) + "__" + df["label"].astype(str)
    train_df, val_df = train_test_split(
        df, test_size=VAL_N, random_state=SEED, stratify=strata
    )
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    print("Recreated deterministic SEED=42 validation")

val_df[["id","category","label","n_images"]].to_csv(
    WORK/"validation_ids.csv", index=False
)

print("Train:", len(train_df), "Val:", len(val_df))
display(pd.crosstab(val_df["category"], val_df["label"], margins=True))

Recreated deterministic SEED=42 validation
Train: 12371 Val: 600


label,0,1,All
category,,,
БАД,88,258,346
Легковоспламеняющиеся,245,9,254
All,333,267,600


In [5]:
# 4. Contact sheets for all rows
def fit_tile(img, box):
    img = img.convert("RGB")
    img.thumbnail(box, Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", box, "white")
    canvas.paste(img, ((box[0]-img.width)//2, (box[1]-img.height)//2))
    return canvas

def make_sheet(paths, out_path):
    if out_path.exists():
        return str(out_path)
    paths = list(paths)[:MAX_IMAGES]
    if not paths:
        Image.new("RGB",(CONTACT_SHEET_SIZE,CONTACT_SHEET_SIZE),"white").save(
            out_path, "JPEG", quality=JPEG_QUALITY
        )
        return str(out_path)

    n = len(paths)
    if n == 1: cols, rows = 1,1
    elif n <= 4: cols, rows = 2, math.ceil(n/2)
    else: cols, rows = 3,2

    gap = 4
    tw = (CONTACT_SHEET_SIZE-gap*(cols-1))//cols
    th = (CONTACT_SHEET_SIZE-gap*(rows-1))//rows
    sheet = Image.new("RGB",(CONTACT_SHEET_SIZE,CONTACT_SHEET_SIZE),"white")

    for i,p in enumerate(paths):
        try:
            with Image.open(p) as im:
                tile = fit_tile(im,(tw,th))
        except:
            tile = Image.new("RGB",(tw,th),"white")
        sheet.paste(tile, ((i%cols)*(tw+gap),(i//cols)*(th+gap)))

    sheet.save(out_path,"JPEG",quality=JPEG_QUALITY,optimize=False)
    return str(out_path)

all_df = pd.concat([
    train_df.assign(split="train"),
    val_df.assign(split="val")
], ignore_index=True)

def work_sheet(row):
    p = SHEETS_DIR/f"{id_str(row.id)}.jpg"
    return row.Index, make_sheet(row.image_paths,p)

paths = [None]*len(all_df)
with ThreadPoolExecutor(max_workers=16) as ex:
    futures = [ex.submit(work_sheet,row) for row in all_df.itertuples()]
    for f in tqdm(as_completed(futures), total=len(futures), desc="Contact sheets"):
        i,p = f.result()
        paths[i] = p

all_df["sheet_path"] = paths
MANIFEST = WORK/"train_manifest.csv"
all_df[[
    "id","name","description","category","label","n_images","split","sheet_path"
]].to_csv(MANIFEST,index=False)

display(all_df.groupby(["split","category","label"]).size().rename("n").reset_index())
disk_status("After sheets")

Contact sheets:   0%|          | 0/12971 [00:00<?, ?it/s]

,split,category,label,n
0,train,БАД,0,1817
1,train,БАД,1,5306
2,train,Легковоспламеняющиеся,0,5059
3,train,Легковоспламеняющиеся,1,189
4,val,БАД,0,88
5,val,БАД,1,258
6,val,Легковоспламеняющиеся,0,245
7,val,Легковоспламеняющиеся,1,9


After sheets: used=0.78 GB free=18.72 GB


In [6]:
# 5. Download Qwen once
MODEL_PATH = snapshot_download(
    repo_id=MODEL_ID,
    cache_dir=str(HF_ROOT/"hub"),
    max_workers=4,
)

xet = HF_ROOT/"xet"
if xet.exists():
    shutil.rmtree(xet, ignore_errors=True)
    xet.mkdir(parents=True, exist_ok=True)

print("MODEL_PATH:", MODEL_PATH)
disk_status("After model")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

MODEL_PATH: /kaggle/working/hf_cache/hub/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a
After model: used=9.48 GB free=10.02 GB


In [7]:
# 6. Write trainer script
TRAIN_SCRIPT = WORK/"train_qwen_qlora.py"
TRAIN_SCRIPT.write_text('\nimport argparse, gc, json, math, random, time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nfrom sklearn.metrics import f1_score, accuracy_score, confusion_matrix\n\nimport torch\nfrom torch.nn.utils import clip_grad_norm_\nfrom transformers import (\n    AutoProcessor,\n    BitsAndBytesConfig,\n    Qwen3_5ForConditionalGeneration,\n    get_cosine_schedule_with_warmup,\n)\nfrom peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\n\nBAD_RULES = """Правила БАД:\n- относится, если есть прямое указание БАД / биологически активная добавка / dietary supplement;\n- спортивное питание при прямом указании на спортпит не относится;\n- если явно сказано, что товар не является БАД, он не относится;\n- без маркировки БАД / dietary supplement товар не относится."""\n\nFIRE_RULES = """Правила Легковоспламеняющиеся:\n- относится: самостоятельный источник воспламенения; содержит горючее вещество/ЛВЖ/горючий газ; опасный товар входит в комплект;\n- не относится: устройство лишь используется с огнем/топливом, но не содержит его;\n- не относится: горючее содержимое отсутствует в поставке;\n- не относится: источник воспламенения встроен;\n- не относится: горючий материал только компонент;\n- не относится: опасный предмет не входит в комплект."""\n\ndef args_parser():\n    p = argparse.ArgumentParser()\n    p.add_argument("--model_path", required=True)\n    p.add_argument("--manifest", required=True)\n    p.add_argument("--category", required=True)\n    p.add_argument("--output_dir", required=True)\n    p.add_argument("--r", type=int, default=32)\n    p.add_argument("--alpha", type=int, default=64)\n    p.add_argument("--dropout", type=float, default=0.05)\n    p.add_argument("--lr", type=float, default=1e-4)\n    p.add_argument("--epochs", type=int, default=1)\n    p.add_argument("--grad_accum", type=int, default=8)\n    p.add_argument("--weight_decay", type=float, default=0.01)\n    p.add_argument("--warmup_ratio", type=float, default=0.05)\n    p.add_argument("--max_description_chars", type=int, default=2200)\n    p.add_argument("--balanced_sampling", type=int, default=1)\n    p.add_argument("--seed", type=int, default=42)\n    p.add_argument("--log_every", type=int, default=25)\n    p.add_argument("--smoke_only", action="store_true")\n    return p.parse_args()\n\ndef set_seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\ndef trim_desc(value, max_chars):\n    s = "" if pd.isna(value) else str(value)\n    if len(s) <= max_chars:\n        return s\n    h = int(max_chars * 0.70)\n    return s[:h] + "\\n...[середина сокращена]...\\n" + s[-(max_chars-h):]\n\ndef build_prompt(row, max_chars):\n    cat = str(row["category"])\n    rules = BAD_RULES if cat == "БАД" else FIRE_RULES\n    name = "" if pd.isna(row["name"]) else str(row["name"])\n    desc = trim_desc(row["description"], max_chars)\n    return f"""Ты решаешь бинарную классификацию товара.\n\n{rules}\n\nНазвание:\n{name}\n\nОписание:\n{desc}\n\nНа изображении объединены все фотографии товара.\n\nПредскажи целевую метку из обучающей разметки.\nОтветь строго одним символом: 0 или 1.\n\nОтвет:"""\n\ndef prepare_processor(model_path):\n    processor = AutoProcessor.from_pretrained(model_path, local_files_only=True)\n    processor.tokenizer.padding_side = "left"\n    if processor.tokenizer.pad_token_id is None:\n        processor.tokenizer.pad_token = processor.tokenizer.eos_token\n    try:\n        processor.image_processor.size["longest_edge"] = 576 * 576\n        processor.image_processor.size["shortest_edge"] = 224 * 224\n    except Exception:\n        pass\n    z = processor.tokenizer.encode("0", add_special_tokens=False)\n    o = processor.tokenizer.encode("1", add_special_tokens=False)\n    if len(z) != 1 or len(o) != 1:\n        raise RuntimeError(f"Labels must be single tokens: 0={z}, 1={o}")\n    return processor, z[0], o[0]\n\ndef find_text_linear_modules(model):\n    targets = []\n    for name, module in model.named_modules():\n        if not name.startswith("model.language_model."):\n            continue\n        if "linear" not in module.__class__.__name__.lower():\n            continue\n        if name.endswith("lm_head"):\n            continue\n        targets.append(name)\n    targets = sorted(set(targets))\n    if len(targets) < 50:\n        raise RuntimeError(f"Too few LoRA targets: {len(targets)}")\n    return targets\n\ndef load_model(model_path, args):\n    qcfg = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_quant_type="nf4",\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_compute_dtype=torch.float16,\n    )\n    print("Loading Qwen3.5-4B NF4...", flush=True)\n    model = Qwen3_5ForConditionalGeneration.from_pretrained(\n        model_path,\n        quantization_config=qcfg,\n        device_map={"": 0},\n        attn_implementation="sdpa",\n        local_files_only=True,\n    )\n    model.tie_weights()\n    model.config.use_cache = False\n    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)\n    targets = find_text_linear_modules(model)\n    print("LoRA targets:", len(targets), flush=True)\n    lcfg = LoraConfig(\n        r=args.r,\n        lora_alpha=args.alpha,\n        lora_dropout=args.dropout,\n        bias="none",\n        target_modules=targets,\n        task_type="CAUSAL_LM",\n    )\n    model = get_peft_model(model, lcfg)\n    model.print_trainable_parameters()\n    return model, targets\n\ndef make_full_inputs(processor, row, label_ids, max_chars):\n    target = str(int(row["label"]))\n    messages = [\n        {"role":"user","content":[\n            {"type":"image"},\n            {"type":"text","text":build_prompt(row, max_chars)},\n        ]},\n        {"role":"assistant","content":[{"type":"text","text":target}]},\n    ]\n    try:\n        text = processor.apply_chat_template(\n            messages, tokenize=False, add_generation_prompt=False, enable_thinking=False\n        )\n    except TypeError:\n        text = processor.apply_chat_template(\n            messages, tokenize=False, add_generation_prompt=False\n        )\n    with Image.open(row["sheet_path"]) as im:\n        image = im.convert("RGB").copy()\n    inputs = processor(text=[text], images=[image], padding=False, return_tensors="pt")\n    target_id = label_ids[int(row["label"])]\n    ids = inputs["input_ids"][0]\n    pos = (ids == target_id).nonzero(as_tuple=False).flatten()\n    if len(pos) == 0:\n        raise RuntimeError(f"Target token not found for id={row[\'id\']}")\n    labels = torch.full_like(inputs["input_ids"], -100)\n    labels[0, int(pos[-1])] = target_id\n    inputs["labels"] = labels\n    return {k:v.to("cuda:0") for k,v in inputs.items() if isinstance(v, torch.Tensor)}\n\ndef make_prompt_inputs(processor, row, max_chars):\n    messages = [{"role":"user","content":[\n        {"type":"image"},\n        {"type":"text","text":build_prompt(row, max_chars)},\n    ]}]\n    try:\n        text = processor.apply_chat_template(\n            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False\n        )\n    except TypeError:\n        text = processor.apply_chat_template(\n            messages, tokenize=False, add_generation_prompt=True\n        )\n    with Image.open(row["sheet_path"]) as im:\n        image = im.convert("RGB").copy()\n    inputs = processor(text=[text], images=[image], padding=False, return_tensors="pt")\n    return {k:v.to("cuda:0") for k,v in inputs.items() if isinstance(v, torch.Tensor)}\n\ndef balanced_indices(df, seed, enabled):\n    rng = np.random.default_rng(seed)\n    n = len(df)\n    if not enabled:\n        idx = np.arange(n)\n        rng.shuffle(idx)\n        return idx\n    counts = df["label"].value_counts().to_dict()\n    weights = np.array([1.0 / counts[int(y)] for y in df["label"]], dtype=np.float64)\n    weights /= weights.sum()\n    return rng.choice(np.arange(n), size=n, replace=True, p=weights)\n\ndef smoke(model, processor, train_df, label_ids, args):\n    print("=== QLORA SMOKE ===", flush=True)\n    state = {\n        n:p.detach().cpu().clone()\n        for n,p in model.named_parameters() if p.requires_grad\n    }\n    rows = []\n    for label in [0,1]:\n        part = train_df[train_df["label"] == label]\n        if len(part):\n            rows.append(part.iloc[0])\n    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=args.lr)\n    opt.zero_grad(set_to_none=True)\n    losses = []\n    for row in rows:\n        inputs = make_full_inputs(processor, row, label_ids, args.max_description_chars)\n        with torch.autocast("cuda", dtype=torch.float16):\n            out = model(**inputs, use_cache=False)\n            loss = out.loss\n        if not torch.isfinite(loss):\n            raise RuntimeError(f"Non-finite smoke loss {loss.item()}")\n        loss.backward()\n        losses.append(float(loss.detach().cpu()))\n    clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)\n    opt.step()\n    with torch.no_grad():\n        params = dict(model.named_parameters())\n        for n,v in state.items():\n            params[n].copy_(v.to(params[n].device, dtype=params[n].dtype))\n    del state, opt\n    gc.collect()\n    torch.cuda.empty_cache()\n    print("Smoke losses:", losses, flush=True)\n    print("QLORA SMOKE PASSED", flush=True)\n\ndef train(model, processor, train_df, label_ids, args, out_dir):\n    trainable = [p for p in model.parameters() if p.requires_grad]\n    opt = torch.optim.AdamW(\n        trainable, lr=args.lr, weight_decay=args.weight_decay, betas=(0.9,0.999)\n    )\n    updates_per_epoch = math.ceil(len(train_df) / args.grad_accum)\n    total_updates = updates_per_epoch * args.epochs\n    warmup = max(1, int(total_updates * args.warmup_ratio))\n    sched = get_cosine_schedule_with_warmup(opt, warmup, total_updates)\n    opt.zero_grad(set_to_none=True)\n    logs = []\n    micro = 0\n    update = 0\n    t0 = time.time()\n\n    for epoch in range(args.epochs):\n        model.train()\n        idxs = balanced_indices(train_df, args.seed + epoch, bool(args.balanced_sampling))\n        recent = []\n        for j, idx in enumerate(idxs):\n            row = train_df.iloc[int(idx)]\n            inputs = make_full_inputs(processor, row, label_ids, args.max_description_chars)\n            with torch.autocast("cuda", dtype=torch.float16):\n                out = model(**inputs, use_cache=False)\n                raw_loss = out.loss\n                if not torch.isfinite(raw_loss):\n                    raise RuntimeError(f"Non-finite loss id={row[\'id\']}")\n                loss = raw_loss / args.grad_accum\n            loss.backward()\n            recent.append(float(raw_loss.detach().cpu()))\n            micro += 1\n\n            if micro % args.grad_accum == 0 or j == len(idxs)-1:\n                clip_grad_norm_(trainable, 1.0)\n                opt.step()\n                sched.step()\n                opt.zero_grad(set_to_none=True)\n                update += 1\n\n            if micro % args.log_every == 0 or j == len(idxs)-1:\n                rec = {\n                    "epoch":epoch+1,\n                    "micro_step":micro,\n                    "update_step":update,\n                    "loss":float(np.mean(recent[-args.log_every:])),\n                    "lr":float(opt.param_groups[0]["lr"]),\n                    "elapsed_min":(time.time()-t0)/60,\n                }\n                logs.append(rec)\n                print(\n                    f"[{args.category}] epoch {epoch+1}/{args.epochs} "\n                    f"{j+1}/{len(idxs)} loss={rec[\'loss\']:.4f} "\n                    f"lr={rec[\'lr\']:.2e} {rec[\'elapsed_min\']:.1f}m",\n                    flush=True,\n                )\n            del inputs, out, raw_loss, loss\n\n        pd.DataFrame(logs).to_csv(out_dir/"training_log.csv", index=False)\n\n@torch.inference_mode()\ndef evaluate(model, processor, val_df, zero_id, one_id, args):\n    model.eval()\n    rows = []\n    for i,(_,row) in enumerate(val_df.iterrows()):\n        inputs = make_prompt_inputs(processor, row, args.max_description_chars)\n        with torch.autocast("cuda", dtype=torch.float16):\n            out = model(**inputs, use_cache=False, logits_to_keep=1)\n        logits = out.logits[:, -1, :].float()\n        probs = torch.softmax(logits[:, [zero_id, one_id]], dim=-1)[0]\n        p0, p1 = float(probs[0].cpu()), float(probs[1].cpu())\n        rows.append({\n            "id":row["id"], "category":row["category"], "label":int(row["label"]),\n            "p0":p0, "p1":p1, "pred_05":int(p1 >= 0.5),\n        })\n        if (i+1) % 50 == 0:\n            print(f"[{args.category}] val {i+1}/{len(val_df)}", flush=True)\n        del inputs, out, logits, probs\n\n    pred = pd.DataFrame(rows)\n    f1_05 = float(f1_score(pred["label"], pred["pred_05"], zero_division=0))\n    acc = float(accuracy_score(pred["label"], pred["pred_05"]))\n    thresholds = np.linspace(0.05,0.95,181)\n    scores = [\n        float(f1_score(pred["label"], (pred["p1"] >= t).astype(int), zero_division=0))\n        for t in thresholds\n    ]\n    k = int(np.argmax(scores))\n    best_t, best_f1 = float(thresholds[k]), float(scores[k])\n    pred["pred_best"] = (pred["p1"] >= best_t).astype(int)\n    summary = {\n        "n_val":int(len(pred)),\n        "f1_at_0_5":f1_05,\n        "accuracy_at_0_5":acc,\n        "best_threshold":best_t,\n        "best_f1":best_f1,\n        "confusion_matrix_at_0_5":confusion_matrix(\n            pred["label"], pred["pred_05"], labels=[0,1]\n        ).tolist(),\n    }\n    return pred, summary\n\ndef main():\n    args = args_parser()\n    set_seed(args.seed)\n    out_dir = Path(args.output_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    df = pd.read_csv(args.manifest)\n    df = df[df["category"] == args.category].copy()\n    train_df = df[df["split"] == "train"].reset_index(drop=True)\n    val_df = df[df["split"] == "val"].reset_index(drop=True)\n\n    print("CATEGORY:", args.category, flush=True)\n    print("TRAIN:", len(train_df), "VAL:", len(val_df), flush=True)\n    print(train_df["label"].value_counts().sort_index(), flush=True)\n\n    processor, zero_id, one_id = prepare_processor(args.model_path)\n    model, targets = load_model(args.model_path, args)\n    label_ids = {0:zero_id, 1:one_id}\n\n    smoke(model, processor, train_df, label_ids, args)\n    if args.smoke_only:\n        print("SMOKE_ONLY DONE", flush=True)\n        return\n\n    train(model, processor, train_df, label_ids, args, out_dir)\n\n    # Saves adapter only, not the backbone.\n    model.save_pretrained(out_dir, safe_serialization=True)\n\n    pred, val_summary = evaluate(\n        model, processor, val_df, zero_id, one_id, args\n    )\n    pred.to_csv(out_dir/"val_predictions.csv", index=False)\n\n    summary = {\n        "base_model":"Qwen/Qwen3.5-4B",\n        "category":args.category,\n        "target":"direct data.csv label 0/1",\n        "train_rows":int(len(train_df)),\n        "val_rows":int(len(val_df)),\n        "lora":{\n            "r":args.r,\n            "alpha":args.alpha,\n            "dropout":args.dropout,\n            "target_module_count":len(targets),\n            "trainable_params":int(sum(\n                p.numel() for p in model.parameters() if p.requires_grad\n            )),\n        },\n        "training":{\n            "lr":args.lr,\n            "epochs":args.epochs,\n            "grad_accum":args.grad_accum,\n            "balanced_sampling":bool(args.balanced_sampling),\n            "max_description_chars":args.max_description_chars,\n        },\n        "validation":val_summary,\n    }\n    (out_dir/"target_modules.json").write_text(\n        json.dumps(targets, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    (out_dir/"training_summary.json").write_text(\n        json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print("=== ADAPTER SAVED ===", out_dir, flush=True)\n    print(json.dumps(summary, ensure_ascii=False, indent=2), flush=True)\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
print(TRAIN_SCRIPT, TRAIN_SCRIPT.stat().st_size, "bytes")

/kaggle/working/ecup_phase3/train_qwen_qlora.py 16557 bytes


## Smoke test

Один реальный QLoRA forward/backward на GPU0. Если здесь проблема с 4-bit, PEFT, processor или backward — полный train не стартует.

In [8]:
# 6.5 FIX Qwen3.5 fast kernels

!pip uninstall -y fla-core flash-linear-attention causal-conv1d >/dev/null 2>&1 || true

!pip install -U "flash-linear-attention[cuda]"
!pip install -U causal-conv1d --no-build-isolation

  Using cached flash_linear_attention-0.5.2-py3-none-any.whl.metadata (45 kB)
  Using cached fla_core-0.5.2-py3-none-any.whl.metadata (45 kB)
Using cached fla_core-0.5.2-py3-none-any.whl (819 kB)
Using cached flash_linear_attention-0.5.2-py3-none-any.whl (399 kB)
  Using cached causal_conv1d-1.6.2.post1.tar.gz (29 kB)
  Preparing metadata (pyproject.toml) ... done
  Created wheel for causal-conv1d: filename=causal_conv1d-1.6.2.post1-cp312-cp312-linux_x86_64.whl size=193835395 sha256=c16c1c48d4fa63415cc797e02d69f97248c57c04627d99e394d5bb0ef266e288
  Stored in directory: /root/.cache/pip/wheels/ea/f0/26/5d87ae05a302e6dc8016c50cd8c7ee779585f593b9580e7cf8
Successfully built causal-conv1d


In [9]:
# 6.6 VERIFY kernels

import importlib.util
import importlib.metadata as md

assert importlib.util.find_spec("fla") is not None, "fla НЕ установлен"
assert importlib.util.find_spec("causal_conv1d") is not None, "causal_conv1d НЕ установлен"

print("flash-linear-attention:", md.version("flash-linear-attention"))
print("fla-core:", md.version("fla-core"))
print("causal-conv1d:", md.version("causal-conv1d"))

print("=== FAST KERNEL PACKAGES OK ===")

flash-linear-attention: 0.5.2
fla-core: 0.5.2
causal-conv1d: 1.6.2.post1
=== FAST KERNEL PACKAGES OK ===


In [10]:
# 6.7 MEMORY PATCH FOR T4

# Для первого нормального запуска уменьшаем adapter,
# но всё ещё оставляем LoRA на всех text-linear слоях.
LORA_R = 16
LORA_ALPHA = 32

# Картинки на диске НЕ пересоздаём.
# Processor сам уменьшит 576x576 -> максимум ~448x448.
TRAINER_TEXT = TRAIN_SCRIPT.read_text(encoding="utf-8")

# 1. Меньше visual tokens
TRAINER_TEXT = TRAINER_TEXT.replace(
    'processor.image_processor.size["longest_edge"] = 576 * 576',
    'processor.image_processor.size["longest_edge"] = 448 * 448'
)

# 2. Явно грузим НЕ-квантизованные части модели в FP16
TRAINER_TEXT = TRAINER_TEXT.replace(
    'quantization_config=qcfg,\n        device_map={"": 0},',
    'quantization_config=qcfg,\n        dtype=torch.float16,\n        device_map={"": 0},'
)

# 3. gradient checkpointing без reentrant input-grad hook
TRAINER_TEXT = TRAINER_TEXT.replace(
    'model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)',
    '''model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )'''
)

# 4. PEFT после prepare_model_for_kbit_training апкастит
# крупные frozen параметры в FP32. На T4 возвращаем frozen
# embeddings/vision/прочие ненормировочные параметры обратно в FP16.
needle = '''model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
    targets = find_text_linear_modules(model)'''

replacement = '''model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )

    # Memory-critical для T4:
    # frozen non-quantized параметры держим FP16,
    # нормы оставляем FP32 для стабильности.
    for name, param in model.named_parameters():
        if (
            not param.requires_grad
            and param.dtype == torch.float32
            and param.__class__.__name__ != "Params4bit"
            and "norm" not in name.lower()
        ):
            param.data = param.data.to(torch.float16)

    model.tie_weights()
    torch.cuda.empty_cache()

    print(
        "GPU after kbit prep:",
        round(torch.cuda.memory_allocated() / 2**30, 2),
        "GB allocated"
    )

    targets = find_text_linear_modules(model)'''

assert needle in TRAINER_TEXT, "Не найден блок prepare_model_for_kbit_training"

TRAINER_TEXT = TRAINER_TEXT.replace(needle, replacement)

TRAIN_SCRIPT.write_text(
    TRAINER_TEXT,
    encoding="utf-8"
)

print("Patched:", TRAIN_SCRIPT)
print("LORA_R =", LORA_R)
print("LORA_ALPHA =", LORA_ALPHA)
print("max image processor area = 448x448")

Patched: /kaggle/working/ecup_phase3/train_qwen_qlora.py
LORA_R = 16
LORA_ALPHA = 32
max image processor area = 448x448


In [11]:
# 7. Smoke
SMOKE_DIR = WORK/"smoke"
shutil.rmtree(SMOKE_DIR, ignore_errors=True)

cmd = [
    "python","-u",str(TRAIN_SCRIPT),
    "--model_path",MODEL_PATH,
    "--manifest",str(MANIFEST),
    "--category","БАД",
    "--output_dir",str(SMOKE_DIR),
    "--r",str(LORA_R),
    "--alpha",str(LORA_ALPHA),
    "--dropout",str(LORA_DROPOUT),
    "--lr",str(LR),
    "--epochs","1",
    "--grad_accum","1",
    "--max_description_chars",str(MAX_DESCRIPTION_CHARS),
    "--seed",str(SEED),
    "--smoke_only",
]
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

rc = subprocess.run(cmd, env=env).returncode
assert rc == 0, f"Smoke failed: {rc}"
shutil.rmtree(SMOKE_DIR, ignore_errors=True)
print("=== QLORA SMOKE PASSED ===")

CATEGORY: БАД
TRAIN: 7123 VAL: 346
label
0    1817
1    5306
Name: count, dtype: int64
Loading Qwen3.5-4B NF4...


Loading weights: 100%|██████████| 723/723 [00:09<00:00, 79.33it/s] 


GPU after kbit prep: 3.07 GB allocated
LoRA targets: 248
trainable params: 32,464,896 || all params: 4,571,730,432 || trainable%: 0.7101
=== QLORA SMOKE ===
Smoke losses: [0.7876251935958862, 0.48926645517349243]
QLORA SMOKE PASSED
SMOKE_ONLY DONE
=== QLORA SMOKE PASSED ===


## Full QLoRA — две категории параллельно

In [12]:
# 8. Train both adapters
BAD_DIR = ADAPTERS_DIR/"qwen35_4b_BAD_qlora"
FIRE_DIR = ADAPTERS_DIR/"qwen35_4b_FIRE_qlora"
shutil.rmtree(BAD_DIR, ignore_errors=True)
shutil.rmtree(FIRE_DIR, ignore_errors=True)

def make_cmd(category, out_dir):
    return [
        "python","-u",str(TRAIN_SCRIPT),
        "--model_path",MODEL_PATH,
        "--manifest",str(MANIFEST),
        "--category",category,
        "--output_dir",str(out_dir),
        "--r",str(LORA_R),
        "--alpha",str(LORA_ALPHA),
        "--dropout",str(LORA_DROPOUT),
        "--lr",str(LR),
        "--epochs",str(EPOCHS),
        "--grad_accum",str(GRAD_ACCUM),
        "--max_description_chars",str(MAX_DESCRIPTION_CHARS),
        "--balanced_sampling","1" if BALANCED_SAMPLING else "0",
        "--seed",str(SEED),
        "--log_every","25",
    ]

env0 = os.environ.copy()
env0["CUDA_VISIBLE_DEVICES"] = "0"
env0["TOKENIZERS_PARALLELISM"] = "false"

env1 = os.environ.copy()
env1["CUDA_VISIBLE_DEVICES"] = "1"
env1["TOKENIZERS_PARALLELISM"] = "false"

print("GPU0 -> БАД")
p0 = subprocess.Popen(make_cmd("БАД",BAD_DIR), env=env0)

print("GPU1 -> Легковоспламеняющиеся")
p1 = subprocess.Popen(make_cmd("Легковоспламеняющиеся",FIRE_DIR), env=env1)

rc0 = p0.wait()
rc1 = p1.wait()

print("BAD rc:",rc0,"FIRE rc:",rc1)
assert rc0 == 0, "BAD training failed"
assert rc1 == 0, "FIRE training failed"
print("=== BOTH TRAININGS FINISHED ===")

GPU0 -> БАД
GPU1 -> Легковоспламеняющиеся
CATEGORY: БАД
TRAIN: 7123 VAL: 346
label
0    1817
1    5306
Name: count, dtype: int64
CATEGORY: Легковоспламеняющиеся
TRAIN: 5248 VAL: 254
label
0    5059
1     189
Name: count, dtype: int64
Loading Qwen3.5-4B NF4...
Loading Qwen3.5-4B NF4...


Loading weights: 100%|██████████| 723/723 [00:13<00:00, 53.67it/s] 


GPU after kbit prep: 3.08 GB allocated
LoRA targets: 248
GPU after kbit prep: 3.08 GB allocated
LoRA targets: 248
trainable params: 32,464,896 || all params: 4,571,730,432 || trainable%: 0.7101
=== QLORA SMOKE ===
trainable params: 32,464,896 || all params: 4,571,730,432 || trainable%: 0.7101
=== QLORA SMOKE ===
Smoke losses: [0.1691066473722458, 0.3192494809627533]
QLORA SMOKE PASSED
Smoke losses: [0.7876251935958862, 0.48926645517349243]
QLORA SMOKE PASSED
[Легковоспламеняющиеся] epoch 1/1 25/5248 loss=0.6745 lr=9.38e-06 1.2m
[БАД] epoch 1/1 25/7123 loss=0.6842 lr=6.82e-06 1.5m
[Легковоспламеняющиеся] epoch 1/1 50/5248 loss=0.5528 lr=1.88e-05 2.5m
[БАД] epoch 1/1 50/7123 loss=0.4123 lr=1.36e-05 3.1m
[Легковоспламеняющиеся] epoch 1/1 75/5248 loss=0.4729 lr=2.81e-05 3.9m
[БАД] epoch 1/1 75/7123 loss=0.3187 lr=2.05e-05 4.5m
[Легковоспламеняющиеся] epoch 1/1 100/5248 loss=0.6830 lr=3.75e-05 5.3m
[БАД] epoch 1/1 100/7123 loss=0.6281 lr=2.73e-05 6.0m
[Легковоспламеняющиеся] epoch 1/1 125/5

In [13]:
# 9. Results
def load_json(path):
    with open(path,"r",encoding="utf-8") as f:
        return json.load(f)

bad = load_json(BAD_DIR/"training_summary.json")
fire = load_json(FIRE_DIR/"training_summary.json")

table = pd.DataFrame([
    {
        "category":"БАД",
        "train_rows":bad["train_rows"],
        "val_rows":bad["val_rows"],
        "r":bad["lora"]["r"],
        "trainable_params":bad["lora"]["trainable_params"],
        "f1@0.5":bad["validation"]["f1_at_0_5"],
        "best_threshold":bad["validation"]["best_threshold"],
        "best_f1":bad["validation"]["best_f1"],
    },
    {
        "category":"Легковоспламеняющиеся",
        "train_rows":fire["train_rows"],
        "val_rows":fire["val_rows"],
        "r":fire["lora"]["r"],
        "trainable_params":fire["lora"]["trainable_params"],
        "f1@0.5":fire["validation"]["f1_at_0_5"],
        "best_threshold":fire["validation"]["best_threshold"],
        "best_f1":fire["validation"]["best_f1"],
    },
])

display(table)
print("Mean F1 @0.5:", float(table["f1@0.5"].mean()))
print("Mean tuned F1:", float(table["best_f1"].mean()))
table.to_csv(WORK/"phase3_qwen_qlora_summary.csv",index=False)

,category,train_rows,val_rows,r,trainable_params,f1@0.5,best_threshold,best_f1
0,БАД,7123,346,16,32464896,0.938856,0.85,0.942346
1,Легковоспламеняющиеся,5248,254,16,32464896,0.823529,0.05,0.823529


Mean F1 @0.5: 0.8811927137718993
Mean tuned F1: 0.8829376681089931


In [14]:
# 10. Verify adapter files
for folder in [BAD_DIR,FIRE_DIR]:
    print("\n",folder.name)
    for p in sorted(folder.rglob("*")):
        if p.is_file():
            print(p.relative_to(folder), f"{p.stat().st_size/2**20:.2f} MB")
    assert (folder/"adapter_config.json").exists()
    assert list(folder.glob("adapter_model*.safetensors"))
print("\nAdapters verified.")


 qwen35_4b_BAD_qlora
README.md 0.01 MB
adapter_config.json 0.00 MB
adapter_model.safetensors 123.92 MB
target_modules.json 0.01 MB
training_log.csv 0.02 MB
training_summary.json 0.00 MB
val_predictions.csv 0.02 MB

 qwen35_4b_FIRE_qlora
README.md 0.01 MB
adapter_config.json 0.00 MB
adapter_model.safetensors 123.92 MB
target_modules.json 0.01 MB
training_log.csv 0.01 MB
training_summary.json 0.00 MB
val_predictions.csv 0.02 MB

Adapters verified.


In [15]:
# 11. ZIP adapters for download
BAD_ZIP = Path(shutil.make_archive(
    "/kaggle/working/qwen35_BAD_qlora_adapter",
    "zip",
    root_dir=str(BAD_DIR),
))
FIRE_ZIP = Path(shutil.make_archive(
    "/kaggle/working/qwen35_FIRE_qlora_adapter",
    "zip",
    root_dir=str(FIRE_DIR),
))

combined = Path("/kaggle/working/qwen35_phase3_adapters")
shutil.rmtree(combined, ignore_errors=True)
combined.mkdir(parents=True)

shutil.copytree(BAD_DIR, combined/"BAD")
shutil.copytree(FIRE_DIR, combined/"FIRE")
shutil.copy2(WORK/"validation_ids.csv", combined/"validation_ids.csv")
shutil.copy2(WORK/"phase3_qwen_qlora_summary.csv", combined/"phase3_qwen_qlora_summary.csv")

COMBINED_ZIP = Path(shutil.make_archive(
    "/kaggle/working/qwen35_phase3_adapters",
    "zip",
    root_dir=str(combined),
))

for p in [BAD_ZIP,FIRE_ZIP,COMBINED_ZIP]:
    print(p, f"{p.stat().st_size/2**20:.2f} MB")
    display(FileLink(str(p)))

print("СКАЧАЙ ZIP ДО ЗАВЕРШЕНИЯ KAGGLE SESSION.")

/kaggle/working/qwen35_BAD_qlora_adapter.zip 114.83 MB


/kaggle/working/qwen35_BAD_qlora_adapter.zip

/kaggle/working/qwen35_FIRE_qlora_adapter.zip 114.73 MB


/kaggle/working/qwen35_FIRE_qlora_adapter.zip

/kaggle/working/qwen35_phase3_adapters.zip 229.56 MB


/kaggle/working/qwen35_phase3_adapters.zip

СКАЧАЙ ZIP ДО ЗАВЕРШЕНИЯ KAGGLE SESSION.


In [16]:
# 12. Final saved summary
phase3 = {
    "base_model":MODEL_ID,
    "target":"direct data.csv label 0/1",
    "validation_n":int(len(val_df)),
    "contact_sheet_size":CONTACT_SHEET_SIZE,
    "lora_r":LORA_R,
    "lora_alpha":LORA_ALPHA,
    "lr":LR,
    "epochs":EPOCHS,
    "grad_accum":GRAD_ACCUM,
    "bad":bad,
    "fire":fire,
    "mean_f1_at_0_5":float(table["f1@0.5"].mean()),
    "mean_best_f1":float(table["best_f1"].mean()),
}
with open("/kaggle/working/phase3_result.json","w",encoding="utf-8") as f:
    json.dump(phase3,f,ensure_ascii=False,indent=2)
print(json.dumps(phase3,ensure_ascii=False,indent=2))

{
  "base_model": "Qwen/Qwen3.5-4B",
  "target": "direct data.csv label 0/1",
  "validation_n": 600,
  "contact_sheet_size": 576,
  "lora_r": 16,
  "lora_alpha": 32,
  "lr": 0.0001,
  "epochs": 1,
  "grad_accum": 8,
  "bad": {
    "base_model": "Qwen/Qwen3.5-4B",
    "category": "БАД",
    "target": "direct data.csv label 0/1",
    "train_rows": 7123,
    "val_rows": 346,
    "lora": {
      "r": 16,
      "alpha": 32,
      "dropout": 0.05,
      "target_module_count": 248,
      "trainable_params": 32464896
    },
    "training": {
      "lr": 0.0001,
      "epochs": 1,
      "grad_accum": 8,
      "balanced_sampling": true,
      "max_description_chars": 2200
    },
    "validation": {
      "n_val": 346,
      "f1_at_0_5": 0.9388560157790927,
      "accuracy_at_0_5": 0.9104046242774566,
      "best_threshold": 0.8499999999999999,
      "best_f1": 0.9423459244532804,
      "confusion_matrix_at_0_5": [
        [
          77,
          11
        ],
        [
          20,
          

In [17]:
# 13. OPTIONAL cleanup — только после скачивания ZIP
# shutil.rmtree(HF_ROOT, ignore_errors=True)
# shutil.rmtree(SHEETS_DIR, ignore_errors=True)
# disk_status("After cleanup")

In [18]:
from pathlib import Path
import shutil
import json
import zipfile
import textwrap
from IPython.display import FileLink, display

# ============================================================
# E-CUP 2026 — SUBMIT V1
# Qwen3.5-4B + BAD LoRA + FIRE LoRA
# Optimized direct-logit inference for H100 80GB
# ============================================================

WORKING = Path("/kaggle/working")

BAD_SRC = Path(
    "/kaggle/working/ecup_phase3/adapters/qwen35_4b_BAD_qlora"
)
FIRE_SRC = Path(
    "/kaggle/working/ecup_phase3/adapters/qwen35_4b_FIRE_qlora"
)

assert BAD_SRC.exists(), BAD_SRC
assert FIRE_SRC.exists(), FIRE_SRC

for folder in [BAD_SRC, FIRE_SRC]:
    assert (folder / "adapter_model.safetensors").exists(), folder
    assert (folder / "adapter_config.json").exists(), folder

SUBMIT_DIR = WORKING / "ecup_qwen35_lora_submit_v1"
shutil.rmtree(SUBMIT_DIR, ignore_errors=True)

(SUBMIT_DIR / "adapters" / "BAD").mkdir(
    parents=True,
    exist_ok=True,
)
(SUBMIT_DIR / "adapters" / "FIRE").mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Копируем ТОЛЬКО то, что реально нужно для inference.
# Base Qwen НЕ копируем — он будет в /shared_models.
# ============================================================

for src, dst in [
    (BAD_SRC, SUBMIT_DIR / "adapters" / "BAD"),
    (FIRE_SRC, SUBMIT_DIR / "adapters" / "FIRE"),
]:
    shutil.copy2(
        src / "adapter_model.safetensors",
        dst / "adapter_model.safetensors",
    )

    shutil.copy2(
        src / "adapter_config.json",
        dst / "adapter_config.json",
    )


# ============================================================
# metadata.json
# ============================================================

metadata = {
    "image": "odsai/ecup26-quality-baseline:1.0",
    "entry_point": "python -u run.py",
}

with open(
    SUBMIT_DIR / "metadata.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# run.py
# ============================================================

RUN_PY = r'''
import os

# BEFORE torch import
os.environ.setdefault(
    "PYTORCH_ALLOC_CONF",
    "expandable_segments:True",
)
os.environ.setdefault(
    "TOKENIZERS_PARALLELISM",
    "false",
)

import argparse
import gc
import json
import math
import re
import time

from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from PIL import Image, ImageFile
from safetensors.torch import load_file
from transformers import AutoProcessor


# ------------------------------------------------------------
# Model class compatibility
# ------------------------------------------------------------

try:
    from transformers import Qwen3_5ForConditionalGeneration

    MODEL_CLASS = Qwen3_5ForConditionalGeneration

except ImportError:

    try:
        from transformers import AutoModelForImageTextToText

        MODEL_CLASS = AutoModelForImageTextToText

    except ImportError:
        from transformers import AutoModelForVision2Seq

        MODEL_CLASS = AutoModelForVision2Seq


ImageFile.LOAD_TRUNCATED_IMAGES = True


# ============================================================
# PATHS
# ============================================================

APP_DIR = Path(__file__).resolve().parent

BAD_ADAPTER = (
    APP_DIR
    / "adapters"
    / "BAD"
)

FIRE_ADAPTER = (
    APP_DIR
    / "adapters"
    / "FIRE"
)


SHARED_MODELS = Path(
    os.environ.get(
        "SHARED_MODELS_PATH",
        "/shared_models",
    )
)

MODEL_PATH = (
    SHARED_MODELS
    / "Qwen"
    / "Qwen3.5-4B"
)


# ============================================================
# INFERENCE CONFIG
# ============================================================

# Exactly as in our fine-tuning pipeline
CONTACT_SHEET_SIZE = 576

# During QLoRA training we patched processor budget to 448x448
PROCESSOR_MAX_SIDE = 448

MAX_IMAGES = 5
MAX_DESCRIPTION_CHARS = 2200

# H100 80 GB:
# direct forward, no generation, no KV cache.
DEFAULT_BATCH_SIZE = 64

CPU_WORKERS = min(
    20,
    max(
        1,
        os.cpu_count() or 1,
    ),
)


# ============================================================
# CATEGORY RULES
# Exactly matching fine-tuning prompt
# ============================================================

BAD_RULES = """Правила категории БАД:
- товар относится к БАД, если в описании или на изображении есть прямое указание БАД / биологически активная добавка / dietary supplement;
- спортивное питание с прямым указанием на спортпит не относится к БАД;
- если явно указано, что товар не является БАД, он не относится к БАД;
- без маркировки БАД / dietary supplement товар не относится к БАД."""


FIRE_RULES = """Правила категории Легковоспламеняющиеся:
- относится: самостоятельный источник воспламенения; содержит горючее вещество, ЛВЖ или горючий газ; либо легковоспламеняющийся товар входит в комплект;
- не относится: устройство лишь используется с огнем/топливом, но не содержит его;
- не относится: горючим является содержимое, отсутствующее в поставке;
- не относится: источник воспламенения встроен в изделие;
- не относится: горючий материал является только компонентом другого изделия;
- не относится: легковоспламеняющийся предмет не входит в комплект."""


# ============================================================
# FAST deterministic comments
#
# Мы НЕ запускаем вторую LLM generation.
# Это экономит огромное количество времени.
# ============================================================

COMMENTS = {

    ("БАД", 1):
        "По названию, описанию и изображениям модель не выявила "
        "достаточных признаков нарушения правил проверки категории "
        "БАД; товар проходит автоматическую проверку.",

    ("БАД", 0):
        "По названию, описанию и изображениям модель выявила признаки "
        "несоответствия правилам проверки категории БАД; товар требует "
        "блокировки.",

    ("Легковоспламеняющиеся", 1):
        "По названию, описанию и изображениям модель не выявила "
        "достаточных признаков нарушения правил для легковоспламеняющихся "
        "товаров; товар проходит автоматическую проверку.",

    ("Легковоспламеняющиеся", 0):
        "По названию, описанию и изображениям модель выявила признаки "
        "несоответствия правилам проверки легковоспламеняющихся товаров; "
        "товар требует блокировки.",
}


# ============================================================
# DATA
# ============================================================

def id_to_str(x):

    if (
        isinstance(
            x,
            (float, np.floating),
        )
        and float(x).is_integer()
    ):
        return str(int(x))

    return str(x)


def trim_description(value):

    text = (
        ""
        if pd.isna(value)
        else str(value)
    )

    if len(text) <= MAX_DESCRIPTION_CHARS:
        return text

    head = int(
        MAX_DESCRIPTION_CHARS
        * 0.70
    )

    tail = (
        MAX_DESCRIPTION_CHARS
        - head
    )

    return (
        text[:head]
        + "\n...[середина описания сокращена]...\n"
        + text[-tail:]
    )


def build_prompt(row):

    category = str(
        row["category"]
    )

    rules = (
        BAD_RULES
        if category == "БАД"
        else FIRE_RULES
    )

    name = (
        ""
        if pd.isna(row["name"])
        else str(row["name"])
    )

    description = trim_description(
        row["description"]
    )

    return f"""Ты решаешь бинарную задачу классификации товара.

{rules}

Название:
{name}

Описание:
{description}

На изображении показаны все фотографии товара, объединенные в один лист.

Предскажи целевую метку из обучающей разметки.
Ответь строго одним символом: 0 или 1.

Ответ:"""


# ============================================================
# IMAGES
# ============================================================

def list_images(
    images_root,
    pid,
):

    folder = (
        images_root
        / id_to_str(pid)
    )

    if not folder.exists():
        return []

    extensions = {
        ".jpg",
        ".jpeg",
        ".png",
        ".webp",
    }

    files = [
        p
        for p in folder.iterdir()
        if (
            p.is_file()
            and p.suffix.lower()
            in extensions
        )
    ]

    def sort_key(p):

        try:
            return (
                0,
                int(p.stem),
            )

        except Exception:
            return (
                1,
                p.name,
            )

    return sorted(
        files,
        key=sort_key,
    )[:MAX_IMAGES]


def fit_tile(
    img,
    box,
):

    img = img.convert("RGB")

    img.thumbnail(
        box,
        Image.Resampling.LANCZOS,
    )

    canvas = Image.new(
        "RGB",
        box,
        "white",
    )

    x = (
        box[0]
        - img.width
    ) // 2

    y = (
        box[1]
        - img.height
    ) // 2

    canvas.paste(
        img,
        (x, y),
    )

    return canvas


def make_sheet(item):

    images_root, pid = item

    paths = list_images(
        images_root,
        pid,
    )

    if not paths:

        return Image.new(
            "RGB",
            (
                CONTACT_SHEET_SIZE,
                CONTACT_SHEET_SIZE,
            ),
            "white",
        )

    n = len(paths)

    if n == 1:

        cols = 1
        rows = 1

    elif n <= 4:

        cols = 2
        rows = math.ceil(
            n / 2
        )

    else:

        cols = 3
        rows = 2

    gap = 4

    tile_w = (
        CONTACT_SHEET_SIZE
        - gap * (cols - 1)
    ) // cols

    tile_h = (
        CONTACT_SHEET_SIZE
        - gap * (rows - 1)
    ) // rows

    sheet = Image.new(
        "RGB",
        (
            CONTACT_SHEET_SIZE,
            CONTACT_SHEET_SIZE,
        ),
        "white",
    )

    for i, path in enumerate(paths):

        try:

            with Image.open(path) as im:

                tile = fit_tile(
                    im,
                    (
                        tile_w,
                        tile_h,
                    ),
                )

        except Exception:

            tile = Image.new(
                "RGB",
                (
                    tile_w,
                    tile_h,
                ),
                "white",
            )

        x = (
            i % cols
        ) * (
            tile_w
            + gap
        )

        y = (
            i // cols
        ) * (
            tile_h
            + gap
        )

        sheet.paste(
            tile,
            (
                x,
                y,
            ),
        )

    return sheet


# ============================================================
# PROCESSOR
# ============================================================

def prepare_processor():

    processor = (
        AutoProcessor
        .from_pretrained(
            MODEL_PATH,
            local_files_only=True,
        )
    )

    processor.tokenizer.padding_side = "left"

    if (
        processor
        .tokenizer
        .pad_token_id
        is None
    ):

        processor.tokenizer.pad_token = (
            processor
            .tokenizer
            .eos_token
        )

    # Same visual budget as QLoRA train.
    try:

        size = (
            processor
            .image_processor
            .size
        )

        if isinstance(
            size,
            dict,
        ):

            size["longest_edge"] = (
                PROCESSOR_MAX_SIDE
                * PROCESSOR_MAX_SIDE
            )

            size["shortest_edge"] = (
                224
                * 224
            )

    except Exception:
        pass

    zero_ids = (
        processor
        .tokenizer
        .encode(
            "0",
            add_special_tokens=False,
        )
    )

    one_ids = (
        processor
        .tokenizer
        .encode(
            "1",
            add_special_tokens=False,
        )
    )

    if (
        len(zero_ids) != 1
        or len(one_ids) != 1
    ):

        raise RuntimeError(
            "Expected single-token labels: "
            f"0={zero_ids}, 1={one_ids}"
        )

    return (
        processor,
        zero_ids[0],
        one_ids[0],
    )


# ============================================================
# MANUAL LORA MERGE
#
# Important:
# - NO peft dependency needed at runtime
# - after merge forward is a normal dense Qwen
# - fastest approach for H100
# ============================================================

def module_candidates(stem):

    seen = set()

    queue = [
        stem,
    ]

    prefixes = [
        "base_model.model.",
        "base_model.",
    ]

    while queue:

        current = queue.pop(0)

        if current in seen:
            continue

        seen.add(
            current
        )

        yield current

        for prefix in prefixes:

            if current.startswith(
                prefix
            ):

                queue.append(
                    current[
                        len(prefix):
                    ]
                )


def resolve_module(
    stem,
    module_map,
):

    for candidate in module_candidates(
        stem
    ):

        if candidate in module_map:

            return (
                candidate,
                module_map[
                    candidate
                ],
            )

    # Last-resort suffix match.
    matches = [
        (
            name,
            module,
        )
        for name, module
        in module_map.items()
        if name.endswith(stem)
    ]

    if len(matches) == 1:
        return matches[0]

    raise KeyError(
        "Cannot resolve LoRA module: "
        + stem
    )


def merge_lora(
    model,
    adapter_dir,
):

    adapter_dir = Path(
        adapter_dir
    )

    config = json.loads(
        (
            adapter_dir
            / "adapter_config.json"
        ).read_text(
            encoding="utf-8"
        )
    )

    state = load_file(
        str(
            adapter_dir
            / "adapter_model.safetensors"
        ),
        device="cpu",
    )

    default_r = int(
        config.get(
            "r",
            16,
        )
    )

    default_alpha = float(
        config.get(
            "lora_alpha",
            default_r,
        )
    )

    use_rslora = bool(
        config.get(
            "use_rslora",
            False,
        )
    )

    fan_in_fan_out = bool(
        config.get(
            "fan_in_fan_out",
            False,
        )
    )

    rank_pattern = (
        config.get(
            "rank_pattern"
        )
        or {}
    )

    alpha_pattern = (
        config.get(
            "alpha_pattern"
        )
        or {}
    )

    module_map = dict(
        model.named_modules()
    )

    A_keys = [
        key
        for key in state.keys()
        if re.search(
            r"\.lora_A(?:\.default)?\.weight$",
            key,
        )
    ]

    if not A_keys:

        raise RuntimeError(
            "No LoRA weights found: "
            + str(adapter_dir)
        )

    merged = 0

    with torch.no_grad():

        for A_key in A_keys:

            stem = re.sub(
                r"\.lora_A(?:\.default)?\.weight$",
                "",
                A_key,
            )

            B_key_1 = (
                stem
                + ".lora_B.weight"
            )

            B_key_2 = (
                stem
                + ".lora_B.default.weight"
            )

            if B_key_1 in state:

                B_key = B_key_1

            elif B_key_2 in state:

                B_key = B_key_2

            else:

                raise KeyError(
                    "Missing LoRA B: "
                    + A_key
                )

            resolved_name, module = (
                resolve_module(
                    stem,
                    module_map,
                )
            )

            if not hasattr(
                module,
                "weight",
            ):

                raise TypeError(
                    "No weight in "
                    + resolved_name
                )

            A = state[A_key]
            B = state[B_key]

            r = int(
                rank_pattern.get(
                    resolved_name,
                    rank_pattern.get(
                        stem,
                        A.shape[0],
                    ),
                )
            )

            alpha = float(
                alpha_pattern.get(
                    resolved_name,
                    alpha_pattern.get(
                        stem,
                        default_alpha,
                    ),
                )
            )

            scale = (
                alpha
                / (
                    math.sqrt(r)
                    if use_rslora
                    else r
                )
            )

            dtype = (
                module
                .weight
                .dtype
            )

            device = (
                module
                .weight
                .device
            )

            # Merge directly on H100.
            A_gpu = A.to(
                device=device,
                dtype=dtype,
                non_blocking=True,
            )

            B_gpu = B.to(
                device=device,
                dtype=dtype,
                non_blocking=True,
            )

            delta = torch.matmul(
                B_gpu,
                A_gpu,
            )

            if fan_in_fan_out:
                delta = delta.T

            if (
                delta.shape
                != module.weight.shape
            ):

                if (
                    delta.T.shape
                    == module.weight.shape
                ):

                    delta = delta.T

                else:

                    raise RuntimeError(
                        f"Shape mismatch "
                        f"{resolved_name}: "
                        f"delta={delta.shape}, "
                        f"weight={module.weight.shape}"
                    )

            module.weight.add_(
                delta,
                alpha=scale,
            )

            merged += 1

            del (
                A_gpu,
                B_gpu,
                delta,
            )

    del state

    gc.collect()
    torch.cuda.empty_cache()

    print(
        f"Merged {merged} LoRA modules "
        f"from {adapter_dir.name}",
        flush=True,
    )

    return merged


# ============================================================
# BASE + MERGED ADAPTER
# ============================================================

def load_merged_model(
    adapter_dir,
):

    # Training QLoRA used FP16 compute.
    # H100 executes FP16 extremely efficiently.
    dtype = torch.float16

    kwargs = dict(
        torch_dtype=dtype,
        local_files_only=True,
        low_cpu_mem_usage=True,
        device_map={
            "": 0,
        },
    )

    try:

        model = (
            MODEL_CLASS
            .from_pretrained(
                MODEL_PATH,
                attn_implementation="sdpa",
                **kwargs,
            )
        )

    except TypeError:

        model = (
            MODEL_CLASS
            .from_pretrained(
                MODEL_PATH,
                **kwargs,
            )
        )

    # Qwen checkpoint has tied embeddings/lm_head.
    model.tie_weights()

    model.config.use_cache = False

    model.eval()

    merge_lora(
        model,
        adapter_dir,
    )

    return model


# ============================================================
# CHAT TEMPLATE
# ============================================================

def make_chat_text(
    processor,
    prompt,
):

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                },
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ]

    try:

        return (
            processor
            .apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        )

    except TypeError:

        return (
            processor
            .apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        )


# ============================================================
# ULTRA-FAST ONE-FORWARD CLASSIFICATION
# ============================================================

@torch.inference_mode()
def infer_batch(
    model,
    processor,
    texts,
    images,
    zero_id,
    one_id,
):

    inputs = processor(
        text=texts,
        images=images,
        padding=True,
        return_tensors="pt",
    ).to(
        "cuda"
    )

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
    ):

        try:

            output = model(
                **inputs,
                use_cache=False,

                # Critical optimization:
                # only last position logits
                logits_to_keep=1,
            )

        except TypeError:

            output = model(
                **inputs,
                use_cache=False,
            )

    logits = (
        output
        .logits[
            :,
            -1,
            :
        ]
    )

    # No softmax needed:
    #
    # P(label=1) >= 0.5
    # <=>
    # logit_1 >= logit_0
    #
    prediction = (
        logits[
            :,
            one_id
        ]
        >=
        logits[
            :,
            zero_id
        ]
    )

    prediction = (
        prediction
        .to(
            torch.int8
        )
        .cpu()
        .numpy()
    )

    del (
        inputs,
        output,
        logits,
    )

    return prediction


# ============================================================
# CATEGORY INFERENCE
# ============================================================

def predict_category(
    df,
    row_indices,
    sheets,
    processor,
    zero_id,
    one_id,
    adapter_dir,
    batch_size,
):

    if not row_indices:
        return {}

    print(
        f"Loading {adapter_dir.name} "
        f"for {len(row_indices)} rows",
        flush=True,
    )

    start_time = time.time()

    model = load_merged_model(
        adapter_dir
    )

    print(
        f"Model ready in "
        f"{time.time() - start_time:.1f}s",
        flush=True,
    )

    # IMPORTANT:
    # longest prompts FIRST.
    #
    # If batch=64 fits the hardest batch,
    # everything later will fit too.
    #
    # It also reduces padding waste.
    order = sorted(
        row_indices,
        key=lambda i: len(
            df.at[
                i,
                "_chat_text",
            ]
        ),
        reverse=True,
    )

    result = {}

    position = 0
    current_batch_size = batch_size

    while position < len(order):

        current_indices = order[
            position:
            position
            + current_batch_size
        ]

        texts = [
            df.at[
                i,
                "_chat_text",
            ]
            for i
            in current_indices
        ]

        images = [
            sheets[i]
            for i
            in current_indices
        ]

        try:

            predictions = infer_batch(
                model,
                processor,
                texts,
                images,
                zero_id,
                one_id,
            )

            for i, prediction in zip(
                current_indices,
                predictions,
            ):

                result[i] = int(
                    prediction
                )

            position += len(
                current_indices
            )

            if (
                position
                % max(
                    current_batch_size * 4,
                    128,
                )
                == 0
                or position
                == len(order)
            ):

                print(
                    f"{adapter_dir.name}: "
                    f"{position}/{len(order)} "
                    f"| batch={current_batch_size}",
                    flush=True,
                )

        except torch.cuda.OutOfMemoryError:

            torch.cuda.empty_cache()

            if current_batch_size <= 1:
                raise

            current_batch_size = max(
                1,
                current_batch_size // 2,
            )

            print(
                "CUDA OOM -> "
                f"batch={current_batch_size}",
                flush=True,
            )

    del model

    gc.collect()
    torch.cuda.empty_cache()

    return result


# ============================================================
# OUTPUT FORMAT
# ============================================================

def format_result(
    category,
    prediction,
):

    # Official baseline mapping:
    # label 1 -> НЕ БАН
    # label 0 -> БАН
    verdict = (
        "не бан"
        if int(prediction) == 1
        else "бан"
    )

    comment = COMMENTS.get(
        (
            category,
            int(prediction),
        )
    )

    if comment is None:

        comment = (
            "По названию, описанию и изображениям "
            "модель выполнила автоматическую проверку "
            "товара по заданным правилам и сформировала "
            "итоговый вердикт."
        )

    if len(comment) < 50:

        comment += (
            " "
            * (
                50
                - len(comment)
            )
        )

    if len(comment) > 300:

        comment = comment[:300]

    return (
        f"<комментарий>"
        f"{comment}"
        f"<вердикт>"
        f"{verdict}"
    )


# ============================================================
# MAIN
# ============================================================

def main():

    parser = argparse.ArgumentParser()

    # Support BOTH spellings used by statement/baseline.
    parser.add_argument(
        "-i",
        "--test_data_path",
        "--test-data-path",
        dest="test_data_path",
        required=True,
    )

    parser.add_argument(
        "-o",
        "--output_path",
        "--output-path",
        dest="output_path",
        required=True,
    )

    args = parser.parse_args()


    # --------------------------------------------------------
    # Hardware checks
    # --------------------------------------------------------

    if not torch.cuda.is_available():

        raise RuntimeError(
            "CUDA GPU is required"
        )

    if not MODEL_PATH.exists():

        raise FileNotFoundError(
            "Shared Qwen not found: "
            + str(MODEL_PATH)
        )


    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


    print(
        "GPU:",
        torch.cuda.get_device_name(0),
        flush=True,
    )

    print(
        "MODEL:",
        MODEL_PATH,
        flush=True,
    )


    # --------------------------------------------------------
    # Input
    # --------------------------------------------------------

    data_path = Path(
        args.test_data_path
    )

    images_root = (
        data_path.parent
        / "images"
    )

    df = (
        pd.read_csv(
            data_path
        )
        .reset_index(
            drop=True
        )
    )


    required_columns = {
        "id",
        "name",
        "description",
        "category",
    }

    missing = (
        required_columns
        - set(df.columns)
    )

    if missing:

        raise ValueError(
            "Missing columns: "
            + str(
                sorted(missing)
            )
        )


    unknown_categories = (
        set(
            df[
                "category"
            ]
            .astype(str)
            .unique()
        )
        -
        {
            "БАД",
            "Легковоспламеняющиеся",
        }
    )

    if unknown_categories:

        raise ValueError(
            "Unknown categories: "
            + str(
                sorted(
                    unknown_categories
                )
            )
        )


    print(
        "Rows:",
        len(df),
        flush=True,
    )


    # --------------------------------------------------------
    # CPU PREPROCESS
    #
    # Competition gives 20 CPU cores.
    # Build all sheets in parallel while GPU is still free.
    # --------------------------------------------------------

    sheet_start = time.time()

    with ThreadPoolExecutor(
        max_workers=CPU_WORKERS
    ) as pool:

        sheets = list(
            pool.map(
                make_sheet,
                [
                    (
                        images_root,
                        product_id,
                    )
                    for product_id
                    in df[
                        "id"
                    ].tolist()
                ],
            )
        )

    print(
        f"Contact sheets: "
        f"{len(sheets)} "
        f"in "
        f"{time.time() - sheet_start:.1f}s",
        flush=True,
    )


    # --------------------------------------------------------
    # Processor + prompts
    # --------------------------------------------------------

    (
        processor,
        zero_id,
        one_id,
    ) = prepare_processor()


    prompts = [
        build_prompt(row)
        for _,
        row
        in df.iterrows()
    ]


    df["_chat_text"] = [
        make_chat_text(
            processor,
            prompt,
        )
        for prompt
        in prompts
    ]


    # --------------------------------------------------------
    # H100 batch
    # --------------------------------------------------------

    batch_size = int(
        os.environ.get(
            "ECUP_BATCH_SIZE",
            DEFAULT_BATCH_SIZE,
        )
    )

    print(
        "Initial batch size:",
        batch_size,
        flush=True,
    )


    predictions = np.ones(
        len(df),
        dtype=np.int8,
    )


    bad_indices = (
        df.index[
            df[
                "category"
            ]
            .astype(str)
            ==
            "БАД"
        ]
        .tolist()
    )


    fire_indices = (
        df.index[
            df[
                "category"
            ]
            .astype(str)
            ==
            "Легковоспламеняющиеся"
        ]
        .tolist()
    )


    # --------------------------------------------------------
    # IMPORTANT SPEED DESIGN
    #
    # We load two fresh dense bases sequentially.
    #
    # Adapter gets MERGED ONCE.
    #
    # Therefore all thousands of forward passes are
    # plain dense Qwen, with zero PEFT overhead.
    #
    # 80GB H100 easily has enough memory for this.
    # --------------------------------------------------------

    for indices, adapter_path in [

        (
            bad_indices,
            BAD_ADAPTER,
        ),

        (
            fire_indices,
            FIRE_ADAPTER,
        ),

    ]:

        pred_map = predict_category(
            df=df,
            row_indices=indices,
            sheets=sheets,
            processor=processor,
            zero_id=zero_id,
            one_id=one_id,
            adapter_dir=adapter_path,
            batch_size=batch_size,
        )

        for i, pred in pred_map.items():

            predictions[i] = pred

            # free already-used image RAM
            sheets[i] = None


    # --------------------------------------------------------
    # Result strings
    # --------------------------------------------------------

    results = [

        format_result(
            str(category),
            int(pred),
        )

        for category, pred
        in zip(
            df["category"],
            predictions,
        )
    ]


    # --------------------------------------------------------
    # STRICT FORMAT GUARD
    # --------------------------------------------------------

    for result in results:

        if (
            not result.startswith(
                "<комментарий>"
            )
            or
            "<вердикт>"
            not in result
        ):

            raise RuntimeError(
                "Invalid result format"
            )

        comment = (
            result
            .split(
                "<комментарий>",
                1,
            )[1]
            .split(
                "<вердикт>",
                1,
            )[0]
        )

        verdict = (
            result
            .split(
                "<вердикт>",
                1,
            )[1]
        )

        if verdict not in {
            "бан",
            "не бан",
        }:

            raise RuntimeError(
                "Invalid verdict: "
                + verdict
            )

        if not (
            50
            <= len(comment)
            <= 300
        ):

            raise RuntimeError(
                "Invalid comment length: "
                + str(
                    len(comment)
                )
            )


    # --------------------------------------------------------
    # SAVE
    # --------------------------------------------------------

    output = pd.DataFrame(
        {
            "id": df["id"],
            "result": results,
        }
    )


    output_path = Path(
        args.output_path
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    output.to_csv(
        output_path,
        index=False,
    )


    print(
        "SAVED:",
        output_path,
        "| rows:",
        len(output),
        flush=True,
    )


if __name__ == "__main__":
    main()
'''


RUN_PATH = SUBMIT_DIR / "run.py"

RUN_PATH.write_text(
    RUN_PY,
    encoding="utf-8",
)


# ============================================================
# BASIC BUILD VALIDATION
# ============================================================

# Python syntax
compile(
    RUN_PY,
    "run.py",
    "exec",
)

# metadata
with open(
    SUBMIT_DIR / "metadata.json",
    "r",
    encoding="utf-8",
) as f:
    loaded_metadata = json.load(f)

assert (
    loaded_metadata["entry_point"]
    == "python -u run.py"
)

# adapter sanity
for folder in [
    SUBMIT_DIR / "adapters" / "BAD",
    SUBMIT_DIR / "adapters" / "FIRE",
]:
    assert (
        folder
        / "adapter_model.safetensors"
    ).exists()

    assert (
        folder
        / "adapter_config.json"
    ).exists()


# ============================================================
# ZIP
# ============================================================

ZIP_BASE = (
    "/kaggle/working/"
    "ecup_qwen35_lora_submit_v1"
)

ZIP_PATH = Path(
    shutil.make_archive(
        ZIP_BASE,
        "zip",
        root_dir=str(
            SUBMIT_DIR
        ),
    )
)


# ============================================================
# VERIFY ZIP STRUCTURE
# ============================================================

with zipfile.ZipFile(
    ZIP_PATH,
    "r",
) as z:

    names = set(
        z.namelist()
    )

    required = {
        "run.py",
        "metadata.json",
        "adapters/BAD/adapter_model.safetensors",
        "adapters/BAD/adapter_config.json",
        "adapters/FIRE/adapter_model.safetensors",
        "adapters/FIRE/adapter_config.json",
    }

    missing = (
        required
        - names
    )

    assert not missing, missing

    print("\nARCHIVE CONTENTS:")

    for name in sorted(names):

        info = z.getinfo(name)

        print(
            f"{name:55s} "
            f"{info.file_size / 2**20:8.2f} MB"
        )


size_mb = (
    ZIP_PATH.stat().st_size
    / 2**20
)

print(
    "\n=========================================="
)

print(
    "SUBMISSION READY"
)

print(
    "=========================================="
)

print(
    "ZIP:",
    ZIP_PATH,
)

print(
    f"SIZE: {size_mb:.2f} MB"
)

assert (
    ZIP_PATH.stat().st_size
    < 5_000_000_000
), "Archive > 5GB!"

print(
    "\nBase Qwen3.5-4B is NOT included."
)

print(
    "Adapters only + run.py + metadata.json."
)

print(
    "\nDownload:"
)

display(
    FileLink(
        str(ZIP_PATH)
    )
)


ARCHIVE CONTENTS:
adapters/                                                   0.00 MB
adapters/BAD/                                               0.00 MB
adapters/BAD/adapter_config.json                            0.00 MB
adapters/BAD/adapter_model.safetensors                    123.92 MB
adapters/FIRE/                                              0.00 MB
adapters/FIRE/adapter_config.json                           0.00 MB
adapters/FIRE/adapter_model.safetensors                   123.92 MB
metadata.json                                               0.00 MB
run.py                                                      0.03 MB

SUBMISSION READY
ZIP: /kaggle/working/ecup_qwen35_lora_submit_v1.zip
SIZE: 229.53 MB

Base Qwen3.5-4B is NOT included.
Adapters only + run.py + metadata.json.

Download:


/kaggle/working/ecup_qwen35_lora_submit_v1.zip

In [19]:
# ============================================================
# FULL DATASET EVAL — 2 GPUs IN PARALLEL
# GPU0 -> BAD adapter
# GPU1 -> FIRE adapter
# ============================================================

from pathlib import Path
import os
import json
import shutil
import subprocess
import textwrap
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from IPython.display import display, FileLink


# ------------------------------------------------------------
# Existing Phase 3 artifacts
# ------------------------------------------------------------

MODEL_PATH_EVAL = str(MODEL_PATH)
MANIFEST_EVAL = str(MANIFEST)

BAD_DIR_EVAL = Path(
    "/kaggle/working/ecup_phase3/adapters/qwen35_4b_BAD_qlora"
)

FIRE_DIR_EVAL = Path(
    "/kaggle/working/ecup_phase3/adapters/qwen35_4b_FIRE_qlora"
)

assert Path(MODEL_PATH_EVAL).exists(), MODEL_PATH_EVAL
assert Path(MANIFEST_EVAL).exists(), MANIFEST_EVAL
assert BAD_DIR_EVAL.exists(), BAD_DIR_EVAL
assert FIRE_DIR_EVAL.exists(), FIRE_DIR_EVAL

FULL_EVAL_DIR = Path(
    "/kaggle/working/ecup_phase3/full_dataset_eval"
)
FULL_EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

INFER_SCRIPT = FULL_EVAL_DIR / "infer_adapter_full.py"


# ============================================================
# Standalone inference script
# ============================================================

SCRIPT = r'''
import argparse
import gc
import json
import os
import time

from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    confusion_matrix,
)

from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen3_5ForConditionalGeneration,
)

from peft import PeftModel


BAD_RULES = """Правила категории БАД:
- товар относится к БАД, если в описании или на изображении есть прямое указание БАД / биологически активная добавка / dietary supplement;
- спортивное питание с прямым указанием на спортпит не относится к БАД;
- если явно указано, что товар не является БАД, он не относится к БАД;
- без маркировки БАД / dietary supplement товар не относится к БАД."""


FIRE_RULES = """Правила категории Легковоспламеняющиеся:
- относится: самостоятельный источник воспламенения; содержит горючее вещество, ЛВЖ или горючий газ; либо легковоспламеняющийся товар входит в комплект;
- не относится: устройство лишь используется с огнем/топливом, но не содержит его;
- не относится: горючим является содержимое, отсутствующее в поставке;
- не относится: источник воспламенения встроен в изделие;
- не относится: горючий материал является только компонентом другого изделия;
- не относится: легковоспламеняющийся предмет не входит в комплект."""


MAX_DESCRIPTION_CHARS = 2200

# Это тот visual budget, с которым обучались адаптеры.
PROCESSOR_MAX_SIDE = 448


def parse_args():

    p = argparse.ArgumentParser()

    p.add_argument(
        "--model_path",
        required=True,
    )

    p.add_argument(
        "--adapter",
        required=True,
    )

    p.add_argument(
        "--manifest",
        required=True,
    )

    p.add_argument(
        "--category",
        required=True,
    )

    p.add_argument(
        "--output",
        required=True,
    )

    p.add_argument(
        "--batch_size",
        type=int,
        default=8,
    )

    return p.parse_args()


def trim_description(value):

    text = (
        ""
        if pd.isna(value)
        else str(value)
    )

    if len(text) <= MAX_DESCRIPTION_CHARS:
        return text

    head = int(
        MAX_DESCRIPTION_CHARS * 0.70
    )

    tail = (
        MAX_DESCRIPTION_CHARS
        - head
    )

    return (
        text[:head]
        + "\n...[середина описания сокращена]...\n"
        + text[-tail:]
    )


def build_prompt(row):

    category = str(
        row["category"]
    )

    rules = (
        BAD_RULES
        if category == "БАД"
        else FIRE_RULES
    )

    name = (
        ""
        if pd.isna(row["name"])
        else str(row["name"])
    )

    description = trim_description(
        row["description"]
    )

    return f"""Ты решаешь бинарную задачу классификации товара.

{rules}

Название:
{name}

Описание:
{description}

На изображении показаны все фотографии товара, объединенные в один лист.

Предскажи целевую метку из обучающей разметки.
Ответь строго одним символом: 0 или 1.

Ответ:"""


def make_chat_text(
    processor,
    row,
):

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                },
                {
                    "type": "text",
                    "text": build_prompt(row),
                },
            ],
        }
    ]

    try:

        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )

    except TypeError:

        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


def prepare_processor(
    model_path,
):

    processor = AutoProcessor.from_pretrained(
        model_path,
        local_files_only=True,
    )

    processor.tokenizer.padding_side = "left"

    if processor.tokenizer.pad_token_id is None:

        processor.tokenizer.pad_token = (
            processor.tokenizer.eos_token
        )

    try:

        size = processor.image_processor.size

        if isinstance(size, dict):

            size["longest_edge"] = (
                PROCESSOR_MAX_SIDE
                * PROCESSOR_MAX_SIDE
            )

            size["shortest_edge"] = (
                224 * 224
            )

    except Exception:
        pass

    zero_ids = processor.tokenizer.encode(
        "0",
        add_special_tokens=False,
    )

    one_ids = processor.tokenizer.encode(
        "1",
        add_special_tokens=False,
    )

    assert len(zero_ids) == 1
    assert len(one_ids) == 1

    return (
        processor,
        zero_ids[0],
        one_ids[0],
    )


def load_model(
    model_path,
    adapter_path,
):

    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    print(
        "Loading base NF4...",
        flush=True,
    )

    model = (
        Qwen3_5ForConditionalGeneration
        .from_pretrained(
            model_path,
            quantization_config=quant,
            dtype=torch.float16,
            device_map={"": 0},
            attn_implementation="sdpa",
            local_files_only=True,
        )
    )

    model.tie_weights()

    model = PeftModel.from_pretrained(
        model,
        adapter_path,
        is_trainable=False,
    )

    model.eval()
    model.config.use_cache = False

    print(
        "Model + adapter ready",
        flush=True,
    )

    return model


@torch.inference_mode()
def infer(
    model,
    processor,
    df,
    zero_id,
    one_id,
    initial_batch_size,
):

    # --------------------------------------------------------
    # Build prompts once
    # --------------------------------------------------------

    texts = [
        make_chat_text(
            processor,
            row,
        )
        for _, row
        in df.iterrows()
    ]

    # Sort by text length.
    # Similar lengths in one batch = less padding.
    order = sorted(
        range(len(df)),
        key=lambda i: len(texts[i]),
        reverse=True,
    )

    results = [None] * len(df)

    batch_size = initial_batch_size
    pos = 0

    start = time.time()

    while pos < len(order):

        idxs = order[
            pos:
            pos + batch_size
        ]

        batch_texts = [
            texts[i]
            for i in idxs
        ]

        images = []

        try:

            for i in idxs:

                path = df.iloc[i]["sheet_path"]

                with Image.open(path) as im:

                    images.append(
                        im.convert("RGB").copy()
                    )

            inputs = processor(
                text=batch_texts,
                images=images,
                padding=True,
                return_tensors="pt",
            ).to(
                "cuda:0"
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):

                try:

                    out = model(
                        **inputs,
                        use_cache=False,
                        logits_to_keep=1,
                    )

                except TypeError:

                    out = model(
                        **inputs,
                        use_cache=False,
                    )

            logits = (
                out.logits[:, -1, :].float()
            )

            pair = logits[
                :,
                [zero_id, one_id]
            ]

            probs = torch.softmax(
                pair,
                dim=-1,
            )

            probs = probs.cpu().numpy()

            for local_idx, row_idx in enumerate(idxs):

                p0 = float(
                    probs[local_idx, 0]
                )

                p1 = float(
                    probs[local_idx, 1]
                )

                results[row_idx] = (
                    p0,
                    p1,
                )

            pos += len(idxs)

            elapsed = time.time() - start

            if (
                pos % max(
                    100,
                    batch_size * 10,
                ) == 0
                or
                pos == len(order)
            ):

                speed = (
                    pos / elapsed
                    if elapsed > 0
                    else 0
                )

                eta = (
                    (len(order) - pos)
                    / max(speed, 1e-9)
                    / 60
                )

                print(
                    f"{pos}/{len(order)} | "
                    f"batch={batch_size} | "
                    f"{speed:.2f} items/s | "
                    f"ETA={eta:.1f} min",
                    flush=True,
                )

            del (
                inputs,
                out,
                logits,
                pair,
                probs,
                images,
            )

        except torch.cuda.OutOfMemoryError:

            torch.cuda.empty_cache()

            if batch_size == 1:
                raise

            batch_size = max(
                1,
                batch_size // 2,
            )

            print(
                "OOM -> new batch size:",
                batch_size,
                flush=True,
            )

    return results


def metrics_block(
    df,
    pred_col,
):

    result = {}

    for split_name in [
        "train",
        "val",
        "all",
    ]:

        if split_name == "all":

            part = df

        else:

            part = df[
                df["split"] == split_name
            ]

        if len(part) == 0:
            continue

        y = part["label"].astype(int)
        p = part[pred_col].astype(int)

        result[split_name] = {
            "n": int(len(part)),
            "f1": float(
                f1_score(
                    y,
                    p,
                    zero_division=0,
                )
            ),
            "accuracy": float(
                accuracy_score(
                    y,
                    p,
                )
            ),
            "confusion_matrix": (
                confusion_matrix(
                    y,
                    p,
                    labels=[0, 1],
                ).tolist()
            ),
        }

    return result


def main():

    args = parse_args()

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    print(
        "GPU:",
        torch.cuda.get_device_name(0),
        flush=True,
    )

    print(
        "CATEGORY:",
        args.category,
        flush=True,
    )

    manifest = pd.read_csv(
        args.manifest
    )

    df = (
        manifest[
            manifest["category"]
            == args.category
        ]
        .copy()
        .reset_index(drop=True)
    )

    print(
        "Rows:",
        len(df),
        flush=True,
    )

    print(
        "Split counts:"
    )

    print(
        df["split"].value_counts(),
        flush=True,
    )

    print(
        "Labels:"
    )

    print(
        df["label"].value_counts().sort_index(),
        flush=True,
    )

    processor, zero_id, one_id = (
        prepare_processor(
            args.model_path
        )
    )

    model = load_model(
        args.model_path,
        args.adapter,
    )

    values = infer(
        model,
        processor,
        df,
        zero_id,
        one_id,
        args.batch_size,
    )

    df["p0"] = [
        x[0]
        for x in values
    ]

    df["p1"] = [
        x[1]
        for x in values
    ]

    # --------------------------------------------------------
    # Standard threshold used in Phase 3
    # --------------------------------------------------------

    df["pred_05"] = (
        df["p1"] >= 0.5
    ).astype(int)

    # --------------------------------------------------------
    # Phase-3 tuned threshold
    #
    # BAD: 0.85
    # FIRE: 0.5 because 0.05 gave same F1 and
    # 0.5 is safer / less overfitted.
    # --------------------------------------------------------

    threshold = (
        0.85
        if args.category == "БАД"
        else 0.50
    )

    df["pred_phase3_threshold"] = (
        df["p1"] >= threshold
    ).astype(int)

    summary = {
        "category": args.category,
        "threshold_phase3": threshold,
        "at_0_5": metrics_block(
            df,
            "pred_05",
        ),
        "at_phase3_threshold": metrics_block(
            df,
            "pred_phase3_threshold",
        ),
    }

    output_path = Path(
        args.output
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    df[
        [
            "id",
            "category",
            "label",
            "split",
            "p0",
            "p1",
            "pred_05",
            "pred_phase3_threshold",
        ]
    ].to_csv(
        output_path,
        index=False,
    )

    summary_path = (
        output_path.parent
        /
        (
            output_path.stem
            + "_summary.json"
        )
    )

    summary_path.write_text(
        json.dumps(
            summary,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        "\nRESULT:"
    )

    print(
        json.dumps(
            summary,
            ensure_ascii=False,
            indent=2,
        )
    )

    del model

    gc.collect()
    torch.cuda.empty_cache()


if __name__ == "__main__":
    main()
'''

INFER_SCRIPT.write_text(
    SCRIPT,
    encoding="utf-8",
)

compile(
    SCRIPT,
    str(INFER_SCRIPT),
    "exec",
)

print("Inference script:", INFER_SCRIPT)


# ============================================================
# Launch both GPUs
# ============================================================

BAD_OUT = FULL_EVAL_DIR / "BAD_full_predictions.csv"
FIRE_OUT = FULL_EVAL_DIR / "FIRE_full_predictions.csv"


def make_cmd(
    category,
    adapter,
    output,
):

    return [
        "python",
        "-u",
        str(INFER_SCRIPT),

        "--model_path",
        MODEL_PATH_EVAL,

        "--manifest",
        MANIFEST_EVAL,

        "--category",
        category,

        "--adapter",
        str(adapter),

        "--output",
        str(output),

        # Start aggressively.
        # Auto-fallback halves it on OOM.
        "--batch_size",
        "8",
    ]


env0 = os.environ.copy()
env0["CUDA_VISIBLE_DEVICES"] = "0"
env0["TOKENIZERS_PARALLELISM"] = "false"
env0["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

env1 = os.environ.copy()
env1["CUDA_VISIBLE_DEVICES"] = "1"
env1["TOKENIZERS_PARALLELISM"] = "false"
env1["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"


print("\n========================================")
print("GPU0 -> БАД")
print("GPU1 -> Легковоспламеняющиеся")
print("========================================\n")


p_bad = subprocess.Popen(
    make_cmd(
        "БАД",
        BAD_DIR_EVAL,
        BAD_OUT,
    ),
    env=env0,
)

p_fire = subprocess.Popen(
    make_cmd(
        "Легковоспламеняющиеся",
        FIRE_DIR_EVAL,
        FIRE_OUT,
    ),
    env=env1,
)


bad_rc = p_bad.wait()
fire_rc = p_fire.wait()


print("\nBAD return code:", bad_rc)
print("FIRE return code:", fire_rc)

assert bad_rc == 0, "BAD inference failed"
assert fire_rc == 0, "FIRE inference failed"


# ============================================================
# Combined results
# ============================================================

bad_pred = pd.read_csv(BAD_OUT)
fire_pred = pd.read_csv(FIRE_OUT)

full_pred = pd.concat(
    [
        bad_pred,
        fire_pred,
    ],
    ignore_index=True,
)

FULL_OUT = (
    FULL_EVAL_DIR
    / "qwen35_qlora_full_dataset_predictions.csv"
)

full_pred.to_csv(
    FULL_OUT,
    index=False,
)


# ============================================================
# Nice summary table
# ============================================================

rows = []

for category, part in full_pred.groupby(
    "category"
):

    for split_name in [
        "train",
        "val",
        "all",
    ]:

        if split_name == "all":

            p = part

        else:

            p = part[
                part["split"]
                == split_name
            ]

        for pred_col, threshold_name in [
            (
                "pred_05",
                "0.5",
            ),
            (
                "pred_phase3_threshold",
                "phase3",
            ),
        ]:

            rows.append(
                {
                    "category": category,
                    "split": split_name,
                    "threshold": threshold_name,
                    "n": len(p),
                    "f1": f1_score(
                        p["label"],
                        p[pred_col],
                        zero_division=0,
                    ),
                    "accuracy": accuracy_score(
                        p["label"],
                        p[pred_col],
                    ),
                }
            )


metrics = pd.DataFrame(rows)

display(metrics)


# ============================================================
# Competition-like means
# ============================================================

print("\n========================================")
print("MEAN F1 BY SPLIT")
print("========================================")

for split_name in [
    "train",
    "val",
    "all",
]:

    for threshold_name in [
        "0.5",
        "phase3",
    ]:

        x = metrics[
            (metrics["split"] == split_name)
            &
            (metrics["threshold"] == threshold_name)
        ]

        mean_f1 = x["f1"].mean()

        print(
            f"{split_name:5s} | "
            f"{threshold_name:6s} | "
            f"mean F1 = {mean_f1:.6f}"
        )


# ============================================================
# Save metrics
# ============================================================

METRICS_OUT = (
    FULL_EVAL_DIR
    / "full_dataset_metrics.csv"
)

metrics.to_csv(
    METRICS_OUT,
    index=False,
)


# ============================================================
# ZIP everything
# ============================================================

ZIP_PATH = Path(
    shutil.make_archive(
        "/kaggle/working/qwen35_qlora_full_dataset_eval",
        "zip",
        root_dir=str(FULL_EVAL_DIR),
    )
)

print("\n========================================")
print("FULL DATASET EVAL FINISHED")
print("========================================")

print("Predictions:", FULL_OUT)
print("Metrics:", METRICS_OUT)
print("ZIP:", ZIP_PATH)

display(
    FileLink(
        str(ZIP_PATH)
    )
)

Inference script: /kaggle/working/ecup_phase3/full_dataset_eval/infer_adapter_full.py

GPU0 -> БАД
GPU1 -> Легковоспламеняющиеся

GPU: Tesla T4
CATEGORY: Легковоспламеняющиеся
GPU: Tesla T4
CATEGORY: БАД
Rows: 5502
Split counts:
split
train    5248
val       254
Name: count, dtype: int64
Labels:
label
0    5304
1     198
Name: count, dtype: int64
Rows: 7469
Split counts:
split
train    7123
val       346
Name: count, dtype: int64
Labels:
label
0    1905
1    5564
Name: count, dtype: int64
Loading base NF4...
Loading base NF4...


Loading weights: 100%|██████████| 723/723 [00:12<00:00, 57.58it/s] 


Model + adapter ready
Model + adapter ready
200/7469 | batch=8 | 0.54 items/s | ETA=222.8 min
200/5502 | batch=8 | 0.52 items/s | ETA=168.8 min


Traceback (most recent call last):
  File "/kaggle/working/ecup_phase3/full_dataset_eval/infer_adapter_full.py", line 729, in <module>
    main()
  File "/kaggle/working/ecup_phase3/full_dataset_eval/infer_adapter_full.py", line 609, in main
    values = infer(
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/ecup_phase3/full_dataset_eval/infer_adapter_full.py", line 379, in infer
    out = model(
          ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.1

KeyboardInterrupt: 

In [20]:
# ============================================================
# PHASE 4 — STRONG TEXT EXPERT
#
# name + description
# Word TF-IDF + Char TF-IDF
# Separate BAD / FIRE models
# Same Phase-3 train/val split
# ============================================================

from pathlib import Path
import os
import re
import gc
import json
import html
import shutil
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    confusion_matrix,
)

import joblib

from IPython.display import display, FileLink

warnings.filterwarnings("ignore")


# ============================================================
# 1. PATHS
# ============================================================

WORK = Path("/kaggle/working/ecup_phase3")
PHASE4 = Path("/kaggle/working/ecup_phase4_text")

shutil.rmtree(PHASE4, ignore_errors=True)
PHASE4.mkdir(parents=True, exist_ok=True)


# Use current notebook MANIFEST if it exists.
try:
    MANIFEST_P4 = Path(str(MANIFEST))
except Exception:
    MANIFEST_P4 = WORK / "train_manifest.csv"


assert MANIFEST_P4.exists(), f"Manifest not found: {MANIFEST_P4}"

print("MANIFEST:", MANIFEST_P4)


QWEN_VAL = {
    "БАД": (
        WORK
        / "adapters"
        / "qwen35_4b_BAD_qlora"
        / "val_predictions.csv"
    ),

    "Легковоспламеняющиеся": (
        WORK
        / "adapters"
        / "qwen35_4b_FIRE_qlora"
        / "val_predictions.csv"
    ),
}

for k, v in QWEN_VAL.items():
    print(k, "Qwen val:", v, "exists =", v.exists())


# ============================================================
# 2. LOAD EXACT SAME SPLIT
# ============================================================

df = pd.read_csv(MANIFEST_P4)

required = {
    "id",
    "name",
    "description",
    "category",
    "label",
    "split",
}

missing = required - set(df.columns)

assert not missing, f"Missing columns: {missing}"


print("\nDataset:")
display(
    df.groupby(
        ["split", "category", "label"]
    ).size().rename("n").reset_index()
)


# ============================================================
# 3. TEXT CLEANING
# ============================================================

TAG_RE = re.compile(r"<[^>]+>")
SPACE_RE = re.compile(r"\s+")


def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x)

    x = html.unescape(x)

    # Description sometimes contains HTML.
    x = TAG_RE.sub(" ", x)

    x = x.replace("\u00a0", " ")

    x = SPACE_RE.sub(" ", x)

    return x.strip()


def make_text(row):

    name = clean_text(
        row["name"]
    )

    description = clean_text(
        row["description"]
    )

    # Title is disproportionately informative,
    # so give it extra weight without inventing features.
    return (
        "__NAME__ "
        + name
        + "\n__NAME__ "
        + name
        + "\n__DESCRIPTION__ "
        + description
    )


print("\nBuilding texts...")

df["_text"] = [
    make_text(row)
    for _, row in df.iterrows()
]

print("DONE")


# ============================================================
# 4. STRONG WORD + CHAR TF-IDF
# ============================================================

def make_vectorizer():

    word = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.997,

        sublinear_tf=True,
        lowercase=True,

        token_pattern=r"(?u)\b[\w\-]+\b",

        max_features=180_000,

        dtype=np.float32,
    )

    char = TfidfVectorizer(
        analyzer="char_wb",

        # Handles:
        # БАД / dietary supplement / газ / бутан,
        # typos, morphology, model names, etc.
        ngram_range=(3, 6),

        min_df=2,

        sublinear_tf=True,
        lowercase=True,

        max_features=260_000,

        dtype=np.float32,
    )

    return FeatureUnion(
        [
            ("word", word),
            ("char", char),
        ],
        n_jobs=2,
    )


# ============================================================
# 5. THRESHOLD SEARCH
# ============================================================

def exact_best_threshold(
    y_true,
    score,
):

    y_true = np.asarray(
        y_true,
        dtype=np.int8,
    )

    score = np.asarray(
        score,
        dtype=np.float64,
    )

    unique = np.unique(score)

    if len(unique) == 1:

        thresholds = unique

    else:

        mids = (
            unique[:-1]
            + unique[1:]
        ) / 2.0

        eps = 1e-9

        thresholds = np.concatenate(
            [
                [unique[0] - eps],
                mids,
                [unique[-1] + eps],
            ]
        )

    best_f1 = -1.0
    best_t = 0.0

    for t in thresholds:

        pred = (
            score >= t
        ).astype(np.int8)

        f1 = f1_score(
            y_true,
            pred,
            zero_division=0,
        )

        if f1 > best_f1:

            best_f1 = float(f1)
            best_t = float(t)

    return best_t, best_f1


# ============================================================
# 6. TRAIN ONE CATEGORY
# ============================================================

C_VALUES = [
    0.20,
    0.35,
    0.50,
    0.75,
    1.00,
    1.50,
    2.00,
    3.00,
    5.00,
]


def train_category(category):

    print("\n")
    print("=" * 70)
    print("CATEGORY:", category)
    print("=" * 70)

    cat_dir = (
        PHASE4
        / (
            "BAD"
            if category == "БАД"
            else "FIRE"
        )
    )

    cat_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    part = (
        df[
            df["category"] == category
        ]
        .copy()
        .reset_index(drop=True)
    )

    train = (
        part[
            part["split"] == "train"
        ]
        .copy()
        .reset_index(drop=True)
    )

    val = (
        part[
            part["split"] == "val"
        ]
        .copy()
        .reset_index(drop=True)
    )


    print(
        "TRAIN:",
        len(train),
        "| VAL:",
        len(val),
    )

    print(
        "Train labels:",
        train["label"]
        .value_counts()
        .sort_index()
        .to_dict(),
    )

    print(
        "Val labels:",
        val["label"]
        .value_counts()
        .sort_index()
        .to_dict(),
    )


    # --------------------------------------------------------
    # Fit vocabulary ONLY on train.
    # Validation text is unseen.
    # --------------------------------------------------------

    vectorizer = make_vectorizer()

    print("\nFitting TF-IDF...")

    X_train = vectorizer.fit_transform(
        train["_text"]
    )

    X_val = vectorizer.transform(
        val["_text"]
    )

    y_train = (
        train["label"]
        .astype(int)
        .values
    )

    y_val = (
        val["label"]
        .astype(int)
        .values
    )


    print(
        "X_train:",
        X_train.shape,
        "| nnz:",
        X_train.nnz,
    )

    print(
        "X_val:",
        X_val.shape,
    )


    # --------------------------------------------------------
    # Candidate grid
    #
    # We test balanced and native prior.
    # No resampling, so unlike FIRE QLoRA this cannot memorize
    # 189 positives through 50/50 repeated sampling.
    # --------------------------------------------------------

    metrics = []
    candidates = {}

    for class_weight in [
        None,
        "balanced",
    ]:

        for C in C_VALUES:

            name = (
                f"svc_C{C}_"
                + (
                    "balanced"
                    if class_weight
                    else "native"
                )
            )

            print(
                "Training",
                name,
                "...",
                end=" ",
                flush=True,
            )

            model = LinearSVC(
                C=C,
                class_weight=class_weight,
                max_iter=20_000,
                dual=True,
                random_state=42,
            )

            model.fit(
                X_train,
                y_train,
            )


            train_score = (
                model.decision_function(
                    X_train
                )
            )

            val_score = (
                model.decision_function(
                    X_val
                )
            )


            train_pred = (
                train_score >= 0.0
            ).astype(int)

            val_pred = (
                val_score >= 0.0
            ).astype(int)


            train_f1 = f1_score(
                y_train,
                train_pred,
                zero_division=0,
            )

            val_f1 = f1_score(
                y_val,
                val_pred,
                zero_division=0,
            )


            best_t, best_f1 = (
                exact_best_threshold(
                    y_val,
                    val_score,
                )
            )


            best_pred = (
                val_score >= best_t
            ).astype(int)


            rec = {
                "model": name,
                "C": C,
                "class_weight": (
                    "balanced"
                    if class_weight
                    else "native"
                ),

                "train_f1_default": float(
                    train_f1
                ),

                "val_f1_default": float(
                    val_f1
                ),

                "best_threshold": float(
                    best_t
                ),

                "val_f1_best": float(
                    best_f1
                ),

                "val_accuracy_default": float(
                    accuracy_score(
                        y_val,
                        val_pred,
                    )
                ),

                "val_accuracy_best": float(
                    accuracy_score(
                        y_val,
                        best_pred,
                    )
                ),
            }

            metrics.append(rec)

            candidates[name] = {
                "model": model,
                "score": val_score,
                "threshold": best_t,
            }

            print(
                f"default={val_f1:.5f} "
                f"best={best_f1:.5f} "
                f"t={best_t:.4f}"
            )


    metrics_df = (
        pd.DataFrame(metrics)
        .sort_values(
            [
                "val_f1_best",
                "val_f1_default",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )


    print("\nTOP CANDIDATES:")
    display(
        metrics_df.head(10)
    )


    # --------------------------------------------------------
    # Primary model
    # --------------------------------------------------------

    best_name = (
        metrics_df.iloc[0]["model"]
    )

    best = candidates[
        best_name
    ]

    best_model = best["model"]
    best_score = best["score"]
    best_threshold = float(
        best["threshold"]
    )


    pred_default = (
        best_score >= 0.0
    ).astype(int)

    pred_best = (
        best_score
        >= best_threshold
    ).astype(int)


    # Sigmoid is NOT claimed to be calibrated probability.
    # We save it as a convenient monotonic score for later fusion.
    pseudo_p1 = expit(
        best_score
    )


    val_predictions = pd.DataFrame(
        {
            "id": val["id"],
            "category": category,
            "label": y_val,

            "text_score": best_score,

            "text_sigmoid": pseudo_p1,

            "text_pred_default": pred_default,

            "text_pred_best": pred_best,
        }
    )


    # --------------------------------------------------------
    # Qwen comparison
    # --------------------------------------------------------

    comparison = None

    qwen_path = QWEN_VAL[
        category
    ]

    if qwen_path.exists():

        qwen = pd.read_csv(
            qwen_path
        )

        # Robust ID join.
        val_predictions["_id"] = (
            val_predictions["id"]
            .astype(str)
        )

        qwen["_id"] = (
            qwen["id"]
            .astype(str)
        )

        comparison = (
            val_predictions
            .merge(
                qwen[
                    [
                        "_id",
                        "p1",
                        "pred_05",
                    ]
                ],
                on="_id",
                how="inner",
                validate="one_to_one",
            )
        )


        assert (
            len(comparison)
            == len(val)
        ), (
            len(comparison),
            len(val),
        )


        comparison[
            "qwen_correct"
        ] = (
            comparison[
                "pred_05"
            ]
            ==
            comparison[
                "label"
            ]
        )


        comparison[
            "text_correct"
        ] = (
            comparison[
                "text_pred_best"
            ]
            ==
            comparison[
                "label"
            ]
        )


        qwen_f1 = f1_score(
            comparison["label"],
            comparison["pred_05"],
            zero_division=0,
        )

        text_f1 = f1_score(
            comparison["label"],
            comparison["text_pred_best"],
            zero_division=0,
        )


        qwen_wrong = int(
            (
                ~comparison[
                    "qwen_correct"
                ]
            ).sum()
        )

        text_wrong = int(
            (
                ~comparison[
                    "text_correct"
                ]
            ).sum()
        )

        both_wrong = int(
            (
                ~comparison[
                    "qwen_correct"
                ]
                &
                ~comparison[
                    "text_correct"
                ]
            ).sum()
        )

        recoverable = int(
            (
                ~comparison[
                    "qwen_correct"
                ]
                &
                comparison[
                    "text_correct"
                ]
            ).sum()
        )

        dangerous = int(
            (
                comparison[
                    "qwen_correct"
                ]
                &
                ~comparison[
                    "text_correct"
                ]
            ).sum()
        )


        print("\n--- QWEN vs TEXT ---")

        print(
            "Qwen F1:",
            round(
                float(qwen_f1),
                6,
            )
        )

        print(
            "Text F1:",
            round(
                float(text_f1),
                6,
            )
        )

        print()

        print(
            "Qwen errors:",
            qwen_wrong,
        )

        print(
            "Text errors:",
            text_wrong,
        )

        print(
            "Both wrong:",
            both_wrong,
        )

        print(
            "Qwen wrong / Text correct:",
            recoverable,
            "<-- VERY IMPORTANT",
        )

        print(
            "Qwen correct / Text wrong:",
            dangerous,
        )


        comparison.drop(
            columns=["_id"],
            inplace=True,
        )

        comparison.to_csv(
            cat_dir
            / "qwen_text_comparison.csv",
            index=False,
        )


    # --------------------------------------------------------
    # Save artifacts
    # --------------------------------------------------------

    joblib.dump(
        vectorizer,
        cat_dir
        / "tfidf.joblib",
        compress=3,
    )

    joblib.dump(
        best_model,
        cat_dir
        / "linear_svc.joblib",
        compress=3,
    )

    metrics_df.to_csv(
        cat_dir
        / "candidate_metrics.csv",
        index=False,
    )

    val_predictions.drop(
        columns=[
            "_id"
        ],
        errors="ignore",
    ).to_csv(
        cat_dir
        / "val_predictions.csv",
        index=False,
    )


    summary = {

        "category": category,

        "train_n": int(
            len(train)
        ),

        "val_n": int(
            len(val)
        ),

        "best_model": best_name,

        "best_threshold": best_threshold,

        "f1_default": float(
            f1_score(
                y_val,
                pred_default,
                zero_division=0,
            )
        ),

        "f1_best": float(
            f1_score(
                y_val,
                pred_best,
                zero_division=0,
            )
        ),

        "accuracy_default": float(
            accuracy_score(
                y_val,
                pred_default,
            )
        ),

        "accuracy_best": float(
            accuracy_score(
                y_val,
                pred_best,
            )
        ),

        "confusion_default": (
            confusion_matrix(
                y_val,
                pred_default,
                labels=[0, 1],
            ).tolist()
        ),

        "confusion_best": (
            confusion_matrix(
                y_val,
                pred_best,
                labels=[0, 1],
            ).tolist()
        ),
    }


    if comparison is not None:

        summary[
            "qwen_comparison"
        ] = {

            "qwen_errors": qwen_wrong,

            "text_errors": text_wrong,

            "both_wrong": both_wrong,

            "qwen_wrong_text_correct": recoverable,

            "qwen_correct_text_wrong": dangerous,
        }


    (
        cat_dir
        / "summary.json"
    ).write_text(
        json.dumps(
            summary,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )


    print("\nSUMMARY:")
    print(
        json.dumps(
            summary,
            ensure_ascii=False,
            indent=2,
        )
    )


    del (
        X_train,
        X_val,
        vectorizer,
        candidates,
    )

    gc.collect()

    return summary


# ============================================================
# 7. RUN BAD + FIRE
# ============================================================

bad = train_category(
    "БАД"
)

fire = train_category(
    "Легковоспламеняющиеся"
)


# ============================================================
# 8. PHASE SUMMARY
# ============================================================

table = pd.DataFrame(
    [
        {
            "category": "БАД",
            "text_f1_default": bad[
                "f1_default"
            ],
            "text_f1_best": bad[
                "f1_best"
            ],
            "qwen_wrong_text_correct": (
                bad
                .get(
                    "qwen_comparison",
                    {}
                )
                .get(
                    "qwen_wrong_text_correct",
                    None,
                )
            ),
        },

        {
            "category": "Легковоспламеняющиеся",
            "text_f1_default": fire[
                "f1_default"
            ],
            "text_f1_best": fire[
                "f1_best"
            ],
            "qwen_wrong_text_correct": (
                fire
                .get(
                    "qwen_comparison",
                    {}
                )
                .get(
                    "qwen_wrong_text_correct",
                    None,
                )
            ),
        },
    ]
)


print("\n")
print("=" * 70)
print("PHASE 4 RESULT")
print("=" * 70)

display(table)

print(
    "Mean TEXT F1 default:",
    table[
        "text_f1_default"
    ].mean()
)

print(
    "Mean TEXT F1 tuned:",
    table[
        "text_f1_best"
    ].mean()
)


phase4_summary = {

    "BAD": bad,
    "FIRE": fire,

    "mean_text_f1_default": float(
        table[
            "text_f1_default"
        ].mean()
    ),

    "mean_text_f1_best": float(
        table[
            "text_f1_best"
        ].mean()
    ),
}


(
    PHASE4
    / "phase4_result.json"
).write_text(
    json.dumps(
        phase4_summary,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 9. ZIP — download immediately
# ============================================================

ZIP_PATH = Path(
    shutil.make_archive(
        "/kaggle/working/ecup_phase4_text",
        "zip",
        root_dir=str(PHASE4),
    )
)


print("\nZIP:", ZIP_PATH)
print(
    "SIZE:",
    round(
        ZIP_PATH.stat().st_size
        / 1024**2,
        2,
    ),
    "MB",
)

display(
    FileLink(
        str(ZIP_PATH)
    )
)

MANIFEST: /kaggle/working/ecup_phase3/train_manifest.csv
БАД Qwen val: /kaggle/working/ecup_phase3/adapters/qwen35_4b_BAD_qlora/val_predictions.csv exists = True
Легковоспламеняющиеся Qwen val: /kaggle/working/ecup_phase3/adapters/qwen35_4b_FIRE_qlora/val_predictions.csv exists = True

Dataset:


,split,category,label,n
0,train,БАД,0,1817
1,train,БАД,1,5306
2,train,Легковоспламеняющиеся,0,5059
3,train,Легковоспламеняющиеся,1,189
4,val,БАД,0,88
5,val,БАД,1,258
6,val,Легковоспламеняющиеся,0,245
7,val,Легковоспламеняющиеся,1,9



Building texts...
DONE


CATEGORY: БАД
TRAIN: 7123 | VAL: 346
Train labels: {0: 1817, 1: 5306}
Val labels: {0: 88, 1: 258}

Fitting TF-IDF...
X_train: (7123, 422307) | nnz: 25319022
X_val: (346, 422307)
Training svc_C0.2_native ... default=0.93878 best=0.94184 t=0.0716
Training svc_C0.35_native ... default=0.94030 best=0.94640 t=-0.0542
Training svc_C0.5_native ... default=0.94030 best=0.94695 t=0.3043
Training svc_C0.75_native ... default=0.94227 best=0.94640 t=-0.0953
Training svc_C1.0_native ... default=0.94227 best=0.94640 t=-0.1108
Training svc_C1.5_native ... default=0.94030 best=0.94640 t=-0.1363
Training svc_C2.0_native ... default=0.94030 best=0.94640 t=-0.1374
Training svc_C3.0_native ... default=0.94403 best=0.94620 t=-0.0798
Training svc_C5.0_native ... default=0.94403 best=0.94640 t=-0.1431
Training svc_C0.2_balanced ... default=0.93256 best=0.94340 t=-0.1171
Training svc_C0.35_balanced ... default=0.93870 best=0.94531 t=0.0923
Training svc_C0.5_balanced ... default=0.938

,model,C,class_weight,train_f1_default,val_f1_default,best_threshold,val_f1_best,val_accuracy_default,val_accuracy_best
0,svc_C3.0_balanced,3.00,balanced,0.989546,0.939163,-0.065198,0.949153,0.907514,0.921965
1,svc_C2.0_balanced,2.00,balanced,0.988294,0.935115,-0.067193,0.949153,0.901734,0.921965
2,svc_C5.0_balanced,5.00,balanced,0.990601,0.946970,-0.066789,0.947368,0.919075,0.919075
3,svc_C1.5_balanced,1.50,balanced,0.987525,0.935115,-0.108959,0.947368,0.901734,0.919075
4,svc_C0.5_native,0.50,native,0.988616,0.940299,0.304257,0.946955,0.907514,0.921965
5,svc_C0.75_balanced,0.75,balanced,0.986184,0.936902,0.187300,0.946535,0.904624,0.921965
6,svc_C5.0_native,5.00,native,0.992849,0.944030,-0.143137,0.946396,0.913295,0.916185
7,svc_C0.75_native,0.75,native,0.989561,0.942272,-0.095260,0.946396,0.910405,0.916185
8,svc_C1.0_native,1.00,native,0.989937,0.942272,-0.110836,0.946396,0.910405,0.916185
9,svc_C0.35_native,0.35,native,0.987306,0.940299,-0.054186,0.946396,0.907514,0.916185



--- QWEN vs TEXT ---
Qwen F1: 0.938856
Text F1: 0.949153

Qwen errors: 31
Text errors: 27
Both wrong: 14
Qwen wrong / Text correct: 17 <-- VERY IMPORTANT
Qwen correct / Text wrong: 13

SUMMARY:
{
  "category": "БАД",
  "train_n": 7123,
  "val_n": 346,
  "best_model": "svc_C3.0_balanced",
  "best_threshold": -0.06519785229818215,
  "f1_default": 0.9391634980988594,
  "f1_best": 0.9491525423728814,
  "accuracy_default": 0.9075144508670521,
  "accuracy_best": 0.9219653179190751,
  "confusion_default": [
    [
      67,
      21
    ],
    [
      11,
      247
    ]
  ],
  "confusion_best": [
    [
      67,
      21
    ],
    [
      6,
      252
    ]
  ],
  "qwen_comparison": {
    "qwen_errors": 31,
    "text_errors": 27,
    "both_wrong": 14,
    "qwen_wrong_text_correct": 17,
    "qwen_correct_text_wrong": 13
  }
}


CATEGORY: Легковоспламеняющиеся
TRAIN: 5248 | VAL: 254
Train labels: {0: 5059, 1: 189}
Val labels: {0: 245, 1: 9}

Fitting TF-IDF...
X_train: (5248, 269705) | nnz: 13

,model,C,class_weight,train_f1_default,val_f1_default,best_threshold,val_f1_best,val_accuracy_default,val_accuracy_best
0,svc_C0.2_balanced,0.20,balanced,0.976744,0.941176,-0.149151,1.0,0.996063,1.0
1,svc_C0.35_balanced,0.35,balanced,0.976744,0.941176,-0.184355,1.0,0.996063,1.0
2,svc_C0.5_balanced,0.50,balanced,0.981818,0.941176,-0.206769,1.0,0.996063,1.0
3,svc_C0.75_balanced,0.75,balanced,0.981818,0.875000,-0.230746,1.0,0.992126,1.0
4,svc_C1.0_balanced,1.00,balanced,0.981818,0.875000,-0.247068,1.0,0.992126,1.0
5,svc_C1.5_balanced,1.50,balanced,0.981818,0.875000,-0.267942,1.0,0.992126,1.0
6,svc_C2.0_balanced,2.00,balanced,0.981818,0.875000,-0.281318,1.0,0.992126,1.0
7,svc_C3.0_balanced,3.00,balanced,0.986945,0.875000,-0.298258,1.0,0.992126,1.0
8,svc_C5.0_balanced,5.00,balanced,0.992126,0.875000,-0.318683,1.0,0.992126,1.0
9,svc_C0.5_native,0.50,native,0.986807,0.800000,-0.539797,1.0,0.988189,1.0



--- QWEN vs TEXT ---
Qwen F1: 0.823529
Text F1: 1.0

Qwen errors: 3
Text errors: 0
Both wrong: 0
Qwen wrong / Text correct: 3 <-- VERY IMPORTANT
Qwen correct / Text wrong: 0

SUMMARY:
{
  "category": "Легковоспламеняющиеся",
  "train_n": 5248,
  "val_n": 254,
  "best_model": "svc_C0.2_balanced",
  "best_threshold": -0.14915118374530031,
  "f1_default": 0.9411764705882353,
  "f1_best": 1.0,
  "accuracy_default": 0.9960629921259843,
  "accuracy_best": 1.0,
  "confusion_default": [
    [
      245,
      0
    ],
    [
      1,
      8
    ]
  ],
  "confusion_best": [
    [
      245,
      0
    ],
    [
      0,
      9
    ]
  ],
  "qwen_comparison": {
    "qwen_errors": 3,
    "text_errors": 0,
    "both_wrong": 0,
    "qwen_wrong_text_correct": 3,
    "qwen_correct_text_wrong": 0
  }
}


PHASE 4 RESULT


,category,text_f1_default,text_f1_best,qwen_wrong_text_correct
0,БАД,0.939163,0.949153,17
1,Легковоспламеняющиеся,0.941176,1.000000,3


Mean TEXT F1 default: 0.9401699843435474
Mean TEXT F1 tuned: 0.9745762711864407

ZIP: /kaggle/working/ecup_phase4_text.zip
SIZE: 11.03 MB


/kaggle/working/ecup_phase4_text.zip

In [22]:
# ============================================================
# E-CUP 2026 — TEXT-ONLY SUBMIT V1
#
# Phase 4 -> leaderboard probe
#
# BAD:
#   TF-IDF word+char
#   LinearSVC C=5.0 balanced
#   threshold = 0
#
# FIRE:
#   TF-IDF word+char
#   LinearSVC C=0.2 balanced
#   threshold = 0
#
# IMPORTANT:
# - retrain on ALL labeled data (train + our validation)
# - NO VLM
# - NO GPU
# - NO images
# - NO joblib/sklearn pickle compatibility problems
# - runtime should be extremely fast
# ============================================================

from pathlib import Path
import gzip
import html
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile

import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

from IPython.display import FileLink, display


# ============================================================
# 0. PATHS
# ============================================================

MANIFEST_PATH = Path(
    "/kaggle/working/ecup_phase3/train_manifest.csv"
)

assert MANIFEST_PATH.exists(), MANIFEST_PATH

SUBMIT_DIR = Path(
    "/kaggle/working/ecup_text_only_submit_v1"
)

shutil.rmtree(
    SUBMIT_DIR,
    ignore_errors=True,
)

SUBMIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ARTIFACTS_DIR = (
    SUBMIT_DIR
    / "artifacts"
)

ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 1. LOAD FULL LABELED DATA
# ============================================================

df = pd.read_csv(
    MANIFEST_PATH
)

required = {
    "id",
    "name",
    "description",
    "category",
    "label",
}

missing = required - set(df.columns)

assert not missing, missing

print("FULL LABELED DATA:", len(df))

print(
    df.groupby(
        ["category", "label"]
    )
    .size()
)

print()


# ============================================================
# 2. EXACT SAME TEXT PREPROCESSING AS PHASE 4
# ============================================================

TAG_RE = re.compile(
    r"<[^>]+>"
)

SPACE_RE = re.compile(
    r"\s+"
)


def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x)

    x = html.unescape(x)

    x = TAG_RE.sub(
        " ",
        x,
    )

    x = x.replace(
        "\u00a0",
        " ",
    )

    x = SPACE_RE.sub(
        " ",
        x,
    )

    return x.strip()


def make_text(row):

    name = clean_text(
        row["name"]
    )

    description = clean_text(
        row["description"]
    )

    # Same title weighting as Phase 4.
    return (
        "__NAME__ "
        + name
        + "\n__NAME__ "
        + name
        + "\n__DESCRIPTION__ "
        + description
    )


print("Building text...")

df["_text"] = [
    make_text(row)
    for _, row in df.iterrows()
]

print("DONE\n")


# ============================================================
# 3. VECTORIZERS
# ============================================================

def make_word_vectorizer():

    return TfidfVectorizer(
        analyzer="word",

        ngram_range=(
            1,
            2,
        ),

        min_df=2,

        max_df=0.997,

        sublinear_tf=True,

        lowercase=True,

        token_pattern=r"(?u)\b[\w\-]+\b",

        max_features=180_000,

        dtype=np.float32,
    )


def make_char_vectorizer():

    return TfidfVectorizer(
        analyzer="char_wb",

        ngram_range=(
            3,
            6,
        ),

        min_df=2,

        sublinear_tf=True,

        lowercase=True,

        max_features=260_000,

        dtype=np.float32,
    )


# ============================================================
# 4. PORTABLE EXPORT
#
# No sklearn pickle.
#
# Store:
# - vocabulary
# - IDF
# - SVC coefficients
# ============================================================

def save_vocab(path, vocab):

    # sklearn может хранить индексы как np.int64,
    # json их напрямую не сериализует
    vocab = {
        str(token): int(index)
        for token, index in vocab.items()
    }

    with gzip.open(
        path,
        "wt",
        encoding="utf-8",
        compresslevel=6,
    ) as f:

        json.dump(
            vocab,
            f,
            ensure_ascii=False,
            separators=(",", ":"),
        )

def load_vocab(
    path,
):

    with gzip.open(
        path,
        "rt",
        encoding="utf-8",
    ) as f:

        return json.load(f)


# ============================================================
# 5. TRAIN FINAL MODEL ON ALL LABELED DATA
# ============================================================

CONFIGS = {

    "БАД": {
        "folder": "BAD",
        "C": 5.0,
        "threshold": 0.0,
    },

    "Легковоспламеняющиеся": {
        "folder": "FIRE",
        "C": 0.2,
        "threshold": 0.0,
    },

}


build_summary = {}


def train_and_export(
    category,
):

    cfg = CONFIGS[
        category
    ]

    folder = (
        ARTIFACTS_DIR
        / cfg["folder"]
    )

    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


    part = (
        df[
            df["category"]
            == category
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    texts = (
        part["_text"]
        .tolist()
    )

    y = (
        part["label"]
        .astype(np.int8)
        .values
    )


    print("=" * 70)

    print(
        "FINAL TRAIN:",
        category
    )

    print("=" * 70)

    print(
        "Rows:",
        len(part)
    )

    print(
        "Labels:",
        part["label"]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    print(
        "C:",
        cfg["C"]
    )


    # --------------------------------------------------------
    # TF-IDF on ALL available labeled products
    # --------------------------------------------------------

    word = make_word_vectorizer()
    char = make_char_vectorizer()


    print(
        "\nFitting WORD TF-IDF...",
        flush=True,
    )

    X_word = word.fit_transform(
        texts
    )


    print(
        "WORD:",
        X_word.shape,
        "| nnz:",
        X_word.nnz,
    )


    print(
        "Fitting CHAR TF-IDF...",
        flush=True,
    )

    X_char = char.fit_transform(
        texts
    )


    print(
        "CHAR:",
        X_char.shape,
        "| nnz:",
        X_char.nnz,
    )


    # --------------------------------------------------------
    # Same FeatureUnion order:
    #
    # [ word features | char features ]
    # --------------------------------------------------------

    X = sp.hstack(
        [
            X_word,
            X_char,
        ],
        format="csr",
        dtype=np.float32,
    )


    print(
        "TOTAL:",
        X.shape,
        "| nnz:",
        X.nnz,
    )


    # --------------------------------------------------------
    # Final SVC
    # --------------------------------------------------------

    model = LinearSVC(
        C=cfg["C"],

        class_weight="balanced",

        max_iter=20_000,

        dual=True,

        random_state=42,
    )


    print(
        "\nTraining final LinearSVC...",
        flush=True,
    )

    model.fit(
        X,
        y,
    )


    # Diagnostic only.
    train_score = (
        model
        .decision_function(
            X
        )
    )

    train_pred = (
        train_score
        >= cfg["threshold"]
    ).astype(
        np.int8
    )

    train_f1 = f1_score(
        y,
        train_pred,
        zero_division=0,
    )

    print(
        "Full-data train F1:",
        train_f1,
    )


    # --------------------------------------------------------
    # Split coefficients back into word / char
    #
    # This means runtime does NOT even need scipy.hstack().
    # score =
    #     X_word @ coef_word
    #   + X_char @ coef_char
    #   + intercept
    # --------------------------------------------------------

    n_word = (
        X_word.shape[1]
    )

    coef = (
        model.coef_[0]
        .astype(np.float32)
    )

    coef_word = (
        coef[:n_word]
    )

    coef_char = (
        coef[n_word:]
    )


    assert (
        len(coef_word)
        == len(
            word.vocabulary_
        )
    )

    assert (
        len(coef_char)
        == len(
            char.vocabulary_
        )
    )


    # --------------------------------------------------------
    # Portable artifacts
    # --------------------------------------------------------

    print(
        "Saving portable artifacts...",
        flush=True,
    )


    save_vocab(
        folder
        / "word_vocab.json.gz",

        word.vocabulary_,
    )


    save_vocab(
        folder
        / "char_vocab.json.gz",

        char.vocabulary_,
    )


    np.savez_compressed(
        folder
        / "weights.npz",

        word_idf=(
            word.idf_
            .astype(np.float32)
        ),

        char_idf=(
            char.idf_
            .astype(np.float32)
        ),

        coef_word=coef_word,

        coef_char=coef_char,

        intercept=np.asarray(
            model.intercept_,
            dtype=np.float32,
        ),

        threshold=np.asarray(
            [
                cfg[
                    "threshold"
                ]
            ],
            dtype=np.float32,
        ),
    )


    config = {
        "category": category,

        "C": cfg["C"],

        "class_weight": "balanced",

        "threshold": cfg[
            "threshold"
        ],

        "n_word": int(
            len(
                word.vocabulary_
            )
        ),

        "n_char": int(
            len(
                char.vocabulary_
            )
        ),

        "word": {
            "analyzer": "word",
            "ngram_range": [
                1,
                2,
            ],
            "max_df": 0.997,
            "sublinear_tf": True,
            "lowercase": True,
            "token_pattern":
                r"(?u)\b[\w\-]+\b",
        },

        "char": {
            "analyzer":
                "char_wb",
            "ngram_range": [
                3,
                6,
            ],
            "sublinear_tf":
                True,
            "lowercase":
                True,
        },
    }


    (
        folder
        / "config.json"
    ).write_text(
        json.dumps(
            config,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )


    # ========================================================
    # 6. PORTABLE RECONSTRUCTION CHECK
    #
    # Recreate vectors from exported vocabulary+idf.
    # The scores MUST equal original sklearn model.
    # ========================================================

    print(
        "Verifying portable representation...",
        flush=True,
    )


    word_vocab2 = load_vocab(
        folder
        / "word_vocab.json.gz"
    )

    char_vocab2 = load_vocab(
        folder
        / "char_vocab.json.gz"
    )


    weights = np.load(
        folder
        / "weights.npz"
    )


    word2 = TfidfVectorizer(
        analyzer="word",

        ngram_range=(
            1,
            2,
        ),

        vocabulary=word_vocab2,

        sublinear_tf=True,

        lowercase=True,

        token_pattern=r"(?u)\b[\w\-]+\b",

        dtype=np.float32,
    )


    word2.idf_ = (
        weights[
            "word_idf"
        ]
    )


    char2 = TfidfVectorizer(
        analyzer="char_wb",

        ngram_range=(
            3,
            6,
        ),

        vocabulary=char_vocab2,

        sublinear_tf=True,

        lowercase=True,

        dtype=np.float32,
    )


    char2.idf_ = (
        weights[
            "char_idf"
        ]
    )


    # 100-product exact test
    sample_texts = texts[
        :min(
            100,
            len(texts),
        )
    ]


    Xw2 = word2.transform(
        sample_texts
    )

    Xc2 = char2.transform(
        sample_texts
    )


    portable_score = (
        np.asarray(
            Xw2
            @ weights[
                "coef_word"
            ]
        ).ravel()

        +

        np.asarray(
            Xc2
            @ weights[
                "coef_char"
            ]
        ).ravel()

        +

        float(
            weights[
                "intercept"
            ][0]
        )
    )


    original_score = (
        model
        .decision_function(
            X[:len(sample_texts)]
        )
    )


    max_diff = float(
        np.max(
            np.abs(
                portable_score
                - original_score
            )
        )
    )


    print(
        "Portable score max diff:",
        max_diff
    )


    assert max_diff < 1e-4, (
        "Portable export mismatch: "
        + str(max_diff)
    )


    print(
        "PORTABLE CHECK PASSED"
    )


    build_summary[
        category
    ] = {
        "rows": int(
            len(part)
        ),

        "C": float(
            cfg["C"]
        ),

        "threshold":
            float(
                cfg[
                    "threshold"
                ]
            ),

        "word_features":
            int(
                X_word.shape[1]
            ),

        "char_features":
            int(
                X_char.shape[1]
            ),

        "full_train_f1":
            float(
                train_f1
            ),

        "portable_max_diff":
            max_diff,
    }


    # --------------------------------------------------------
    # Free RAM before next category
    # --------------------------------------------------------

    del (
        part,
        texts,
        y,
        word,
        char,
        X_word,
        X_char,
        X,
        model,
        train_score,
        train_pred,
        coef,
        coef_word,
        coef_char,
        word2,
        char2,
        Xw2,
        Xc2,
        weights,
    )

    import gc

    gc.collect()

    print()


# ============================================================
# Train sequentially — safer on RAM
# ============================================================

train_and_export(
    "БАД"
)

train_and_export(
    "Легковоспламеняющиеся"
)


# ============================================================
# 7. BUILD INFO
# ============================================================

(
    SUBMIT_DIR
    / "build_info.json"
).write_text(
    json.dumps(
        {
            "name":
                "E-CUP text-only submit v1",

            "training":
                "all labeled competition data",

            "selection":
                "Phase-4 fixed validation",

            "models":
                build_summary,
        },

        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 8. RUNTIME run.py
# ============================================================

RUN_PY = r'''
# ============================================================
# E-CUP 2026 — TEXT ONLY INFERENCE
#
# No GPU.
# No VLM.
# No image preprocessing.
#
# word TF-IDF + char TF-IDF + LinearSVC
# ============================================================

import os

# Avoid pointless BLAS oversubscription.
os.environ.setdefault(
    "OMP_NUM_THREADS",
    "2",
)

os.environ.setdefault(
    "MKL_NUM_THREADS",
    "2",
)

os.environ.setdefault(
    "OPENBLAS_NUM_THREADS",
    "2",
)


import argparse
import gzip
import html
import json
import re
import time

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
)


APP_DIR = (
    Path(__file__)
    .resolve()
    .parent
)

ARTIFACTS = (
    APP_DIR
    / "artifacts"
)


# ============================================================
# TEXT PREPROCESSING
# ============================================================

TAG_RE = re.compile(
    r"<[^>]+>"
)

SPACE_RE = re.compile(
    r"\s+"
)


def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x)

    x = html.unescape(x)

    x = TAG_RE.sub(
        " ",
        x,
    )

    x = x.replace(
        "\u00a0",
        " ",
    )

    x = SPACE_RE.sub(
        " ",
        x,
    )

    return x.strip()


def make_text(
    name,
    description,
):

    name = clean_text(
        name
    )

    description = clean_text(
        description
    )

    return (
        "__NAME__ "
        + name
        + "\n__NAME__ "
        + name
        + "\n__DESCRIPTION__ "
        + description
    )


# ============================================================
# LOAD PORTABLE EXPERT
# ============================================================

def load_vocab(path):

    with gzip.open(
        path,
        "rt",
        encoding="utf-8",
    ) as f:

        return json.load(f)


class TextExpert:

    def __init__(
        self,
        folder,
    ):

        folder = Path(
            folder
        )

        self.config = json.loads(
            (
                folder
                / "config.json"
            ).read_text(
                encoding="utf-8"
            )
        )


        word_vocab = load_vocab(
            folder
            / "word_vocab.json.gz"
        )

        char_vocab = load_vocab(
            folder
            / "char_vocab.json.gz"
        )


        weights = np.load(
            folder
            / "weights.npz"
        )


        # ----------------------------------------------------
        # Reconstruct vectorizers from portable state.
        #
        # No pickled sklearn object.
        # ----------------------------------------------------

        self.word = TfidfVectorizer(
            analyzer="word",

            ngram_range=(
                1,
                2,
            ),

            vocabulary=word_vocab,

            sublinear_tf=True,

            lowercase=True,

            token_pattern=r"(?u)\b[\w\-]+\b",

            dtype=np.float32,
        )


        self.word.idf_ = (
            weights[
                "word_idf"
            ]
        )


        self.char = TfidfVectorizer(
            analyzer="char_wb",

            ngram_range=(
                3,
                6,
            ),

            vocabulary=char_vocab,

            sublinear_tf=True,

            lowercase=True,

            dtype=np.float32,
        )


        self.char.idf_ = (
            weights[
                "char_idf"
            ]
        )


        self.coef_word = (
            weights[
                "coef_word"
            ].astype(
                np.float32,
                copy=False,
            )
        )


        self.coef_char = (
            weights[
                "coef_char"
            ].astype(
                np.float32,
                copy=False,
            )
        )


        self.intercept = float(
            weights[
                "intercept"
            ][0]
        )


        self.threshold = float(
            weights[
                "threshold"
            ][0]
        )


    def predict(
        self,
        texts,
    ):

        # Separate transforms avoid a huge scipy hstack
        # during hidden-test inference.

        X_word = (
            self.word
            .transform(
                texts
            )
        )


        X_char = (
            self.char
            .transform(
                texts
            )
        )


        score = (
            np.asarray(
                X_word
                @
                self.coef_word
            ).ravel()

            +

            np.asarray(
                X_char
                @
                self.coef_char
            ).ravel()

            +

            self.intercept
        )


        pred = (
            score
            >= self.threshold
        ).astype(
            np.int8
        )


        return (
            score,
            pred,
        )


# ============================================================
# VALID OUTPUT COMMENTS
#
# Leaderboard metric uses the verdict.
# Explanation is deterministic to avoid unnecessary LLM cost.
# ============================================================

COMMENT_PASS = (
    "По названию и описанию карточки модель не выявила "
    "достаточных признаков несоответствия правилам проверки; "
    "товар проходит автоматическую классификацию."
)

COMMENT_BAN = (
    "По названию и описанию карточки модель выявила признаки "
    "несоответствия установленным правилам проверки; "
    "для товара сформирован отрицательный вердикт."
)


assert (
    50
    <= len(
        COMMENT_PASS
    )
    <= 300
)

assert (
    50
    <= len(
        COMMENT_BAN
    )
    <= 300
)


def make_result(pred):

    pred = int(
        pred
    )

    # Official baseline mapping:
    # label 1 -> "не бан"
    # label 0 -> "бан"

    if pred == 1:

        return (
            "<комментарий>"
            + COMMENT_PASS
            + "<вердикт>"
            + "не бан"
        )

    return (
        "<комментарий>"
        + COMMENT_BAN
        + "<вердикт>"
        + "бан"
    )


# ============================================================
# MAIN
# ============================================================

def main():

    parser = argparse.ArgumentParser()


    parser.add_argument(
        "-i",
        "--test_data_path",
        "--test-data-path",
        dest="test_data_path",
        required=True,
    )


    parser.add_argument(
        "-o",
        "--output_path",
        "--output-path",
        dest="output_path",
        required=True,
    )


    args = parser.parse_args()


    started = time.time()


    df = (
        pd.read_csv(
            args.test_data_path
        )
        .reset_index(
            drop=True
        )
    )


    required = {
        "id",
        "name",
        "description",
        "category",
    }


    missing = (
        required
        - set(
            df.columns
        )
    )


    if missing:

        raise ValueError(
            "Missing columns: "
            + str(
                sorted(
                    missing
                )
            )
        )


    print(
        "Rows:",
        len(df),
        flush=True,
    )


    # --------------------------------------------------------
    # Build text once
    # --------------------------------------------------------

    texts = [
        make_text(
            name,
            description,
        )

        for name, description
        in zip(
            df["name"],
            df["description"],
        )
    ]


    predictions = np.zeros(
        len(df),
        dtype=np.int8,
    )


    # --------------------------------------------------------
    # Two category experts
    # --------------------------------------------------------

    category_configs = [

        (
            "БАД",
            ARTIFACTS
            / "BAD",
        ),

        (
            "Легковоспламеняющиеся",
            ARTIFACTS
            / "FIRE",
        ),
    ]


    for category, folder in category_configs:

        indices = np.flatnonzero(
            (
                df[
                    "category"
                ]
                .astype(str)
                .values
                ==
                category
            )
        )


        if len(indices) == 0:
            continue


        print(
            category,
            ":",
            len(indices),
            "rows",
            flush=True,
        )


        t0 = time.time()


        expert = TextExpert(
            folder
        )


        category_texts = [
            texts[i]
            for i
            in indices
        ]


        score, pred = (
            expert.predict(
                category_texts
            )
        )


        predictions[
            indices
        ] = pred


        print(
            category,
            "done in",
            round(
                time.time()
                - t0,
                2,
            ),
            "sec",
            "| positive:",
            int(
                pred.sum()
            ),
            flush=True,
        )


    # --------------------------------------------------------
    # Strict category guard
    # --------------------------------------------------------

    expected_categories = {
        "БАД",
        "Легковоспламеняющиеся",
    }


    unknown = (
        set(
            df[
                "category"
            ]
            .astype(str)
            .unique()
        )
        -
        expected_categories
    )


    if unknown:

        raise ValueError(
            "Unexpected categories: "
            + str(
                sorted(
                    unknown
                )
            )
        )


    # --------------------------------------------------------
    # Format results
    # --------------------------------------------------------

    results = [
        make_result(
            pred
        )
        for pred
        in predictions
    ]


    # --------------------------------------------------------
    # Result-stage guards
    # --------------------------------------------------------

    for result in results:

        if (
            not result.startswith(
                "<комментарий>"
            )
            or
            "<вердикт>"
            not in result
        ):

            raise RuntimeError(
                "Invalid result format"
            )


        comment = (
            result
            .split(
                "<комментарий>",
                1,
            )[1]
            .split(
                "<вердикт>",
                1,
            )[0]
        )


        verdict = (
            result
            .split(
                "<вердикт>",
                1,
            )[1]
        )


        if verdict not in {
            "бан",
            "не бан",
        }:

            raise RuntimeError(
                "Invalid verdict"
            )


        if not (
            50
            <= len(comment)
            <= 300
        ):

            raise RuntimeError(
                "Invalid comment length: "
                + str(
                    len(comment)
                )
            )


    output = pd.DataFrame(
        {
            "id":
                df["id"],

            "result":
                results,
        }
    )


    output_path = Path(
        args.output_path
    )


    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    output.to_csv(
        output_path,
        index=False,
    )


    print(
        "Saved:",
        output_path,
        flush=True,
    )


    print(
        "Total inference:",
        round(
            time.time()
            - started,
            2,
        ),
        "sec",
        flush=True,
    )


if __name__ == "__main__":
    main()
'''


RUN_PATH = (
    SUBMIT_DIR
    / "run.py"
)

RUN_PATH.write_text(
    RUN_PY,
    encoding="utf-8",
)

# Syntax check
compile(
    RUN_PY,
    "run.py",
    "exec",
)


# ============================================================
# 9. metadata.json
# ============================================================

metadata = {
    "image":
        "odsai/ecup26-quality-baseline:1.0",

    "entry_point":
        "python -u run.py",
}


(
    SUBMIT_DIR
    / "metadata.json"
).write_text(
    json.dumps(
        metadata,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 10. RUNTIME SMOKE TEST
#
# Run the ACTUAL packaged run.py, not notebook functions.
# ============================================================

SMOKE_INPUT = (
    SUBMIT_DIR
    / "_smoke_input.csv"
)

SMOKE_OUTPUT = (
    SUBMIT_DIR
    / "_smoke_output.csv"
)


smoke_df = pd.concat(
    [
        df[
            df["category"]
            == "БАД"
        ].head(10),

        df[
            df["category"]
            == "Легковоспламеняющиеся"
        ].head(10),
    ],
    ignore_index=True,
)


smoke_df[
    [
        "id",
        "name",
        "description",
        "category",
    ]
].to_csv(
    SMOKE_INPUT,
    index=False,
)


print("\n" + "=" * 70)
print("PACKAGED RUNTIME SMOKE TEST")
print("=" * 70)


result = subprocess.run(
    [
        sys.executable,
        "-u",
        str(
            RUN_PATH
        ),

        "--test_data_path",
        str(
            SMOKE_INPUT
        ),

        "--output_path",
        str(
            SMOKE_OUTPUT
        ),
    ],

    cwd=str(
        SUBMIT_DIR
    ),

    capture_output=True,

    text=True,
)


print(
    result.stdout
)


if result.stderr:

    print(
        "STDERR:"
    )

    print(
        result.stderr
    )


assert (
    result.returncode
    == 0
), (
    "Runtime smoke failed: "
    + result.stderr
)


smoke_result = pd.read_csv(
    SMOKE_OUTPUT
)


assert (
    len(smoke_result)
    == len(smoke_df)
)


assert list(
    smoke_result.columns
) == [
    "id",
    "result",
]


print(
    "SMOKE TEST PASSED"
)


# Remove smoke files from final archive.
SMOKE_INPUT.unlink(
    missing_ok=True
)

SMOKE_OUTPUT.unlink(
    missing_ok=True
)


# ============================================================
# 11. ZIP
# ============================================================

ZIP_PATH = Path(
    shutil.make_archive(
        "/kaggle/working/ecup_text_only_submit_v1",
        "zip",

        root_dir=str(
            SUBMIT_DIR
        ),
    )
)


# ============================================================
# 12. VERIFY FINAL ARCHIVE
# ============================================================

with zipfile.ZipFile(
    ZIP_PATH,
    "r",
) as z:

    names = set(
        z.namelist()
    )


    required_files = {
        "run.py",
        "metadata.json",

        "artifacts/BAD/config.json",
        "artifacts/BAD/weights.npz",
        "artifacts/BAD/word_vocab.json.gz",
        "artifacts/BAD/char_vocab.json.gz",

        "artifacts/FIRE/config.json",
        "artifacts/FIRE/weights.npz",
        "artifacts/FIRE/word_vocab.json.gz",
        "artifacts/FIRE/char_vocab.json.gz",
    }


    missing = (
        required_files
        - names
    )


    assert not missing, missing


    print("\nFINAL ARCHIVE:")
    print("-" * 70)

    for name in sorted(
        names
    ):

        info = z.getinfo(
            name
        )

        print(
            f"{name:55s} "
            f"{info.file_size / 1024**2:8.2f} MB"
        )


size_mb = (
    ZIP_PATH.stat().st_size
    / 1024**2
)


print("\n" + "=" * 70)

print(
    "TEXT-ONLY SUBMISSION READY"
)

print("=" * 70)

print(
    "ZIP:",
    ZIP_PATH
)

print(
    "SIZE:",
    round(
        size_mb,
        2,
    ),
    "MB",
)

print()

print(
    "NO QWEN"
)

print(
    "NO LoRA"
)

print(
    "NO images"
)

print(
    "NO GPU inference"
)

print(
    "Only TF-IDF + LinearSVC"
)

print()

print(
    "BAD  : C=5.0 balanced, threshold=0"
)

print(
    "FIRE : C=0.2 balanced, threshold=0"
)

print()

print(
    "Trained on ALL available labeled data."
)

print(
    "Portable runtime representation verified."
)


assert (
    ZIP_PATH.stat().st_size
    < 5_000_000_000
)


display(
    FileLink(
        str(
            ZIP_PATH
        )
    )
)

FULL LABELED DATA: 12971
category               label
БАД                    0        1905
                       1        5564
Легковоспламеняющиеся  0        5304
                       1         198
dtype: int64

Building text...
DONE

FINAL TRAIN: БАД
Rows: 7469
Labels: {0: 1905, 1: 5564}
C: 5.0

Fitting WORD TF-IDF...
WORD: (7469, 180000) | nnz: 2616602
Fitting CHAR TF-IDF...
CHAR: (7469, 246775) | nnz: 23908987
TOTAL: (7469, 426775) | nnz: 26525589

Training final LinearSVC...
Full-data train F1: 0.9905831220572257
Saving portable artifacts...
Verifying portable representation...
Portable score max diff: 2.6377333652849444e-06
PORTABLE CHECK PASSED

FINAL TRAIN: Легковоспламеняющиеся
Rows: 5502
Labels: {0: 5304, 1: 198}
C: 0.2

Fitting WORD TF-IDF...
WORD: (5502, 100542) | nnz: 1397009
Fitting CHAR TF-IDF...
CHAR: (5502, 178519) | nnz: 12307777
TOTAL: (5502, 279061) | nnz: 13704786

Training final LinearSVC...
Full-data train F1: 0.9777777777777777
Saving portable artifacts...
Ve

/kaggle/working/ecup_text_only_submit_v1.zip

In [23]:
# ============================================================
# PHASE 5 — MULTIMODAL EMBEDDING EXPERT
#
# Qwen/Qwen3-VL-Embedding-2B
#
# GPU0 + GPU1 parallel embedding extraction
# text + ALL original product images (up to 5)
#
# Then:
#   5-fold OOF model selection ONLY on train
#   untouched Phase-3 validation evaluation
#   comparison against Qwen-LoRA
#
# Safe resumable chunks.
# ============================================================

from pathlib import Path
import os
import sys
import gc
import json
import math
import shutil
import subprocess
import time
import warnings

import numpy as np
import pandas as pd

from IPython.display import display, FileLink

warnings.filterwarnings("ignore")


# ============================================================
# 0. CONFIG
# ============================================================

SEED = 42

MODEL_ID = "Qwen/Qwen3-VL-Embedding-2B"

MANIFEST = Path(
    "/kaggle/working/ecup_phase3/train_manifest.csv"
)

PHASE5 = Path(
    "/kaggle/working/ecup_phase5_embedding"
)

PHASE5.mkdir(
    parents=True,
    exist_ok=True,
)

CHUNKS_DIR = PHASE5 / "chunks"
CHUNKS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

REPO_DIR = Path(
    "/kaggle/working/Qwen3-VL-Embedding"
)

HF_CACHE = Path(
    "/kaggle/working/hf_cache"
)

HF_CACHE.mkdir(
    parents=True,
    exist_ok=True,
)

assert MANIFEST.exists(), MANIFEST
assert os.path.exists("/kaggle/input")

print("Manifest:", MANIFEST)


# ============================================================
# 1. DEPENDENCY — DON'T TOUCH TORCH/TRANSFORMERS
#
# Our current environment already runs Qwen3.5.
# Only install qwen-vl-utils if missing.
# ============================================================

try:
    import qwen_vl_utils
    print(
        "qwen-vl-utils already installed:",
        getattr(
            qwen_vl_utils,
            "__version__",
            "unknown"
        )
    )

except ImportError:

    print("Installing qwen-vl-utils...")

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "qwen-vl-utils>=0.0.14",
        ]
    )

    import qwen_vl_utils

    print("qwen-vl-utils installed")


# ============================================================
# 2. OFFICIAL QWEN EMBEDDING REPO
# ============================================================

EMBEDDER_FILE = (
    REPO_DIR
    / "src"
    / "models"
    / "qwen3_vl_embedding.py"
)

if not EMBEDDER_FILE.exists():

    print(
        "Cloning official "
        "Qwen3-VL-Embedding repo..."
    )

    subprocess.check_call(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/QwenLM/Qwen3-VL-Embedding.git",
            str(REPO_DIR),
        ]
    )

assert EMBEDDER_FILE.exists()

print(
    "Official embedder:",
    EMBEDDER_FILE
)


# ============================================================
# 3. FIND / DOWNLOAD MODEL ONCE
# ============================================================

from huggingface_hub import snapshot_download


def find_existing_embedding_model():

    roots = [
        Path("/kaggle/input"),
        HF_CACHE,
        Path("/kaggle/working"),
    ]

    for root in roots:

        if not root.exists():
            continue

        try:

            for p in root.rglob(
                "Qwen3-VL-Embedding-2B"
            ):

                if (
                    p.is_dir()
                    and
                    (p / "config.json").exists()
                ):

                    return p

        except Exception:
            pass

    return None


MODEL_PATH = find_existing_embedding_model()

if MODEL_PATH is None:

    print(
        "\nDownloading",
        MODEL_ID,
        "..."
    )

    MODEL_PATH = Path(
        snapshot_download(
            repo_id=MODEL_ID,
            cache_dir=str(HF_CACHE),
        )
    )

else:

    print(
        "Existing embedding model found."
    )


assert (
    MODEL_PATH
    / "config.json"
).exists()

print(
    "MODEL_PATH:",
    MODEL_PATH
)


# ============================================================
# 4. FIND ORIGINAL IMAGES ROOT
# ============================================================

def find_images_root(root):

    candidates = []

    for p in root.rglob("images"):

        if not p.is_dir():
            continue

        try:

            dirs = [
                x
                for x in list(
                    p.iterdir()
                )[:100]
                if x.is_dir()
            ]

        except Exception:
            continue

        if not dirs:
            continue

        score = (
            sum(
                x.name.isdigit()
                for x in dirs
            )
            /
            len(dirs)
        )

        if score >= 0.7:

            candidates.append(
                (
                    score,
                    p,
                )
            )

    if not candidates:

        raise FileNotFoundError(
            "images/<id> not found"
        )

    return max(
        candidates,
        key=lambda x: x[0],
    )[1]


IMAGES_ROOT = find_images_root(
    Path("/kaggle/input")
)

print(
    "IMAGES_ROOT:",
    IMAGES_ROOT
)


# ============================================================
# 5. PREPARE INPUT TABLE
# ============================================================

df = (
    pd.read_csv(MANIFEST)
    .reset_index(drop=True)
)

required = {
    "id",
    "name",
    "description",
    "category",
    "label",
    "split",
}

missing = required - set(df.columns)

assert not missing, missing


IMG_EXTS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
}


def id_str(x):

    if (
        isinstance(
            x,
            (float, np.floating),
        )
        and float(x).is_integer()
    ):
        return str(int(x))

    return str(x)


def image_sort_key(p):

    try:

        return (
            0,
            int(p.stem),
        )

    except Exception:

        return (
            1,
            p.name,
        )


def get_images(pid):

    folder = (
        IMAGES_ROOT
        / id_str(pid)
    )

    if not folder.exists():
        return []

    paths = sorted(
        [
            p
            for p in folder.iterdir()
            if (
                p.is_file()
                and
                p.suffix.lower()
                in IMG_EXTS
            )
        ],
        key=image_sort_key,
    )

    return [
        str(p)
        for p in paths[:5]
    ]


# Don't feed unlimited marketplace HTML/text.
MAX_DESCRIPTION_CHARS = 4000


def clean_field(x):

    if pd.isna(x):
        return ""

    return str(x).strip()


def trim_description(x):

    x = clean_field(x)

    if len(x) <= MAX_DESCRIPTION_CHARS:
        return x

    head = int(
        MAX_DESCRIPTION_CHARS
        * 0.70
    )

    tail = (
        MAX_DESCRIPTION_CHARS
        - head
    )

    return (
        x[:head]
        + "\n...[середина сокращена]...\n"
        + x[-tail:]
    )


# Category-specific task instructions.
# No label information is leaked.
BAD_INSTRUCTION = (
    "Представь товар для задачи бинарной классификации "
    "контроля качества карточек категории БАД. "
    "Сохрани семантические и визуальные признаки, связанные "
    "с маркировкой БАД или dietary supplement, спортивным "
    "питанием, назначением товара и явными отрицаниями."
)

FIRE_INSTRUCTION = (
    "Представь товар для задачи бинарной классификации "
    "контроля качества легковоспламеняющихся товаров. "
    "Сохрани семантические и визуальные признаки источников "
    "воспламенения, горючих веществ и газов, состава, "
    "комплектности и случаев, когда горючее не входит в товар."
)


def build_text(row):

    return (
        "Название: "
        + clean_field(row["name"])
        + "\nКатегория: "
        + clean_field(row["category"])
        + "\nОписание: "
        + trim_description(
            row["description"]
        )
    )


print("\nFinding original images...")

image_lists = [
    get_images(pid)
    for pid in df["id"]
]

df["_image_paths"] = image_lists
df["_n_images"] = [
    len(x)
    for x in image_lists
]

df["_text"] = [
    build_text(row)
    for _, row in df.iterrows()
]

df["_instruction"] = np.where(
    df["category"].astype(str)
    == "БАД",
    BAD_INSTRUCTION,
    FIRE_INSTRUCTION,
)

df["_row_idx"] = np.arange(
    len(df),
    dtype=np.int32,
)

print(
    "Rows:",
    len(df)
)

print(
    "Images:",
    int(
        df["_n_images"].sum()
    )
)

display(
    df.groupby(
        [
            "split",
            "category",
            "label",
        ]
    )
    .size()
    .rename("n")
    .reset_index()
)


# ============================================================
# 6. SPLIT DATA BETWEEN GPU0/GPU1
#
# Greedy balance by number of images.
# ============================================================

weights = (
    df["_n_images"]
    .clip(lower=1)
    .values
)

order = np.argsort(
    -weights
)

gpu_indices = [
    [],
    [],
]

gpu_load = [
    0,
    0,
]

for i in order:

    gpu = int(
        gpu_load[1]
        < gpu_load[0]
    )

    gpu_indices[gpu].append(
        int(i)
    )

    gpu_load[gpu] += int(
        weights[i]
    )


print(
    "\nGPU0 rows:",
    len(gpu_indices[0]),
    "weighted load:",
    gpu_load[0],
)

print(
    "GPU1 rows:",
    len(gpu_indices[1]),
    "weighted load:",
    gpu_load[1],
)


PART_FILES = []

for gpu in range(2):

    part = df.iloc[
        gpu_indices[gpu]
    ].copy()

    # JSON keeps list of image paths portable.
    part["image_paths_json"] = (
        part["_image_paths"]
        .apply(
            json.dumps
        )
    )

    path = (
        PHASE5
        / f"gpu{gpu}_items.csv"
    )

    part[
        [
            "_row_idx",
            "id",
            "category",
            "label",
            "split",
            "_text",
            "_instruction",
            "image_paths_json",
            "_n_images",
        ]
    ].to_csv(
        path,
        index=False,
    )

    PART_FILES.append(path)


# ============================================================
# 7. STANDALONE GPU WORKER
# ============================================================

WORKER = PHASE5 / "embed_worker.py"


WORKER_CODE = r'''
import argparse
import gc
import json
import os
import sys
import time

from pathlib import Path

import numpy as np
import pandas as pd
import torch


def parse_args():

    p = argparse.ArgumentParser()

    p.add_argument(
        "--items",
        required=True,
    )

    p.add_argument(
        "--model",
        required=True,
    )

    p.add_argument(
        "--repo",
        required=True,
    )

    p.add_argument(
        "--output_dir",
        required=True,
    )

    p.add_argument(
        "--max_pixels",
        type=int,
        default=261120,
    )

    p.add_argument(
        "--initial_batch",
        type=int,
        default=4,
    )

    p.add_argument(
        "--chunk_rows",
        type=int,
        default=64,
    )

    return p.parse_args()


def main():

    args = parse_args()

    sys.path.insert(
        0,
        args.repo,
    )

    from src.models.qwen3_vl_embedding import (
        Qwen3VLEmbedder
    )

    if not torch.cuda.is_available():

        raise RuntimeError(
            "CUDA required"
        )

    print(
        "GPU:",
        torch.cuda.get_device_name(0),
        flush=True,
    )

    items = (
        pd.read_csv(args.items)
        .reset_index(drop=True)
    )

    output_dir = Path(
        args.output_dir
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    # --------------------------------------------------------
    # T4 -> FP16.
    # SDPA avoids dependency on flash-attn.
    # --------------------------------------------------------

    print(
        "Loading embedding model...",
        flush=True,
    )

    t_load = time.time()

    embedder = Qwen3VLEmbedder(
        model_name_or_path=args.model,

        max_length=8192,

        min_pixels=4096,

        max_pixels=args.max_pixels,

        torch_dtype=torch.float16,

        attn_implementation="sdpa",
    )

    print(
        "Model loaded in",
        round(
            time.time() - t_load,
            1,
        ),
        "sec",
        flush=True,
    )


    def make_input(row):

        obj = {
            "text": str(
                row["_text"]
            ),

            "instruction": str(
                row["_instruction"]
            ),
        }

        paths = json.loads(
            row["image_paths_json"]
        )

        if paths:

            obj["image"] = paths

        return obj


    def process_adaptive(
        rows,
        initial_batch,
    ):

        all_embeddings = []

        pos = 0

        batch_size = min(
            initial_batch,
            len(rows),
        )

        while pos < len(rows):

            current = rows.iloc[
                pos:
                pos + batch_size
            ]

            batch_inputs = [
                make_input(row)
                for _, row
                in current.iterrows()
            ]

            try:

                with torch.inference_mode():

                    emb = embedder.process(
                        batch_inputs,
                        normalize=True,
                    )

                emb = (
                    emb.detach()
                    .float()
                    .cpu()
                    .numpy()
                    .astype(
                        np.float16
                    )
                )

                all_embeddings.append(
                    emb
                )

                pos += len(
                    current
                )


            except torch.cuda.OutOfMemoryError:

                del batch_inputs

                gc.collect()
                torch.cuda.empty_cache()

                if batch_size <= 1:

                    raise

                batch_size = max(
                    1,
                    batch_size // 2,
                )

                print(
                    "OOM -> mini batch:",
                    batch_size,
                    flush=True,
                )

        return np.concatenate(
            all_embeddings,
            axis=0,
        )


    started = time.time()

    total_chunks = (
        len(items)
        + args.chunk_rows
        - 1
    ) // args.chunk_rows


    for chunk_num, start in enumerate(
        range(
            0,
            len(items),
            args.chunk_rows,
        )
    ):

        end = min(
            start
            + args.chunk_rows,
            len(items),
        )

        chunk_file = (
            output_dir
            / f"chunk_{start:06d}_{end:06d}.npz"
        )


        # ----------------------------------------------------
        # RESUME: already completed chunk -> skip.
        # ----------------------------------------------------

        if chunk_file.exists():

            try:

                old = np.load(
                    chunk_file
                )

                if (
                    len(old["row_idx"])
                    ==
                    end - start
                ):

                    print(
                        f"chunk "
                        f"{chunk_num + 1}/"
                        f"{total_chunks} "
                        f"already exists",
                        flush=True,
                    )

                    continue

            except Exception:
                pass


        rows = (
            items.iloc[
                start:end
            ]
            .reset_index(
                drop=True
            )
        )


        emb = process_adaptive(
            rows,
            args.initial_batch,
        )


        np.savez(
            chunk_file,

            row_idx=(
                rows["_row_idx"]
                .astype(np.int32)
                .values
            ),

            embeddings=emb,
        )


        done = end

        elapsed = (
            time.time()
            - started
        )

        # Approximate speed in rows/s for THIS process.
        speed = (
            done / elapsed
            if elapsed > 0
            else 0
        )

        remaining = (
            len(items)
            - done
        )

        eta = (
            remaining
            / max(
                speed,
                1e-9,
            )
            / 60
        )


        print(
            f"chunk "
            f"{chunk_num + 1}/"
            f"{total_chunks} "
            f"| rows {start}:{end} "
            f"| {speed:.3f} rows/s "
            f"| ETA ~{eta:.1f}m",
            flush=True,
        )


    print(
        "WORKER FINISHED",
        flush=True,
    )


if __name__ == "__main__":
    main()
'''


WORKER.write_text(
    WORKER_CODE,
    encoding="utf-8",
)

compile(
    WORKER_CODE,
    str(WORKER),
    "exec",
)

print(
    "\nWorker:",
    WORKER
)


# ============================================================
# 8. RUN BOTH T4s IN PARALLEL
#
# max_pixels=261120 = strong quality setting.
# If OOM, mini-batch automatically halves.
# ============================================================

def worker_cmd(gpu):

    return [
        sys.executable,
        "-u",
        str(WORKER),

        "--items",
        str(PART_FILES[gpu]),

        "--model",
        str(MODEL_PATH),

        "--repo",
        str(REPO_DIR),

        "--output_dir",
        str(
            CHUNKS_DIR
            / f"gpu{gpu}"
        ),

        "--max_pixels",
        "261120",

        "--initial_batch",
        "4",

        "--chunk_rows",
        "64",
    ]


env0 = os.environ.copy()

env0[
    "CUDA_VISIBLE_DEVICES"
] = "0"

env0[
    "PYTORCH_ALLOC_CONF"
] = "expandable_segments:True"

env0[
    "TOKENIZERS_PARALLELISM"
] = "false"


env1 = os.environ.copy()

env1[
    "CUDA_VISIBLE_DEVICES"
] = "1"

env1[
    "PYTORCH_ALLOC_CONF"
] = "expandable_segments:True"

env1[
    "TOKENIZERS_PARALLELISM"
] = "false"


print("\n" + "=" * 70)
print("GPU0 + GPU1 MULTIMODAL EMBEDDINGS")
print("=" * 70)


p0 = subprocess.Popen(
    worker_cmd(0),
    env=env0,
)

p1 = subprocess.Popen(
    worker_cmd(1),
    env=env1,
)


rc0 = p0.wait()
rc1 = p1.wait()


print(
    "\nGPU0 return code:",
    rc0
)

print(
    "GPU1 return code:",
    rc1
)


assert rc0 == 0, (
    "GPU0 embedding failed"
)

assert rc1 == 0, (
    "GPU1 embedding failed"
)


# ============================================================
# 9. COMBINE RESUMABLE CHUNKS
# ============================================================

chunk_files = sorted(
    CHUNKS_DIR.rglob(
        "chunk_*.npz"
    )
)

assert chunk_files, (
    "No embedding chunks found"
)


# Infer dimension from first chunk.
first = np.load(
    chunk_files[0]
)

EMBED_DIM = (
    first[
        "embeddings"
    ].shape[1]
)

print(
    "\nEmbedding dimension:",
    EMBED_DIM
)


embeddings = np.zeros(
    (
        len(df),
        EMBED_DIM,
    ),
    dtype=np.float16,
)

seen = np.zeros(
    len(df),
    dtype=bool,
)


for path in chunk_files:

    z = np.load(path)

    idx = (
        z[
            "row_idx"
        ].astype(int)
    )

    emb = z[
        "embeddings"
    ]

    assert (
        len(idx)
        ==
        len(emb)
    )

    embeddings[idx] = emb

    seen[idx] = True


print(
    "Embedded rows:",
    int(
        seen.sum()
    ),
    "/",
    len(seen),
)


assert seen.all(), (
    f"Missing "
    f"{(~seen).sum()} embeddings"
)


EMBED_PATH = (
    PHASE5
    / "qwen3vl_embeddings_fp16.npy"
)

np.save(
    EMBED_PATH,
    embeddings,
)


INDEX_PATH = (
    PHASE5
    / "embedding_index.csv"
)

df[
    [
        "id",
        "category",
        "label",
        "split",
        "_n_images",
    ]
].to_csv(
    INDEX_PATH,
    index=False,
)


print(
    "Embeddings:",
    EMBED_PATH
)

print(
    "Size:",
    round(
        EMBED_PATH.stat().st_size
        / 1024**2,
        2,
    ),
    "MB",
)


# ============================================================
# 10. CLASSIFIER SELECTION
#
# IMPORTANT:
#
# Hyperparameters + threshold are selected with
# 5-fold OOF ONLY ON TRAIN.
#
# Fixed 600 validation stays untouched until final evaluation.
# ============================================================

from sklearn.model_selection import (
    StratifiedKFold,
)

from sklearn.svm import (
    LinearSVC,
)

from sklearn.linear_model import (
    LogisticRegression,
)

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    confusion_matrix,
    precision_recall_curve,
)

import joblib


X_ALL = embeddings.astype(
    np.float32
)


# Embeddings should already be normalized,
# but renormalize after fp16 storage to be safe.
norms = np.linalg.norm(
    X_ALL,
    axis=1,
    keepdims=True,
)

X_ALL /= np.maximum(
    norms,
    1e-8,
)


def best_threshold_from_scores(
    y,
    scores,
):

    y = np.asarray(
        y,
        dtype=np.int8,
    )

    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    precision, recall, thresholds = (
        precision_recall_curve(
            y,
            scores,
        )
    )

    f1 = (
        2
        * precision
        * recall
        /
        np.maximum(
            precision
            + recall,
            1e-12,
        )
    )

    best_idx = int(
        np.nanargmax(f1)
    )

    if best_idx >= len(
        thresholds
    ):

        threshold = float(
            scores.min()
            - 1e-6
        )

    else:

        threshold = float(
            thresholds[
                best_idx
            ]
        )

    return (
        threshold,
        float(
            f1[
                best_idx
            ]
        ),
    )


def make_model(
    family,
    C,
    class_weight,
):

    cw = (
        None
        if class_weight
        == "native"
        else "balanced"
    )

    if family == "svc":

        return LinearSVC(
            C=C,

            class_weight=cw,

            max_iter=30_000,

            random_state=SEED,
        )

    if family == "logreg":

        return LogisticRegression(
            C=C,

            class_weight=cw,

            max_iter=5000,

            solver="liblinear",

            random_state=SEED,
        )

    raise ValueError(
        family
    )


# Compact but strong grid.
CANDIDATES = []


for C in [
    0.03,
    0.10,
    0.30,
    1.00,
    3.00,
]:

    for cw in [
        "native",
        "balanced",
    ]:

        CANDIDATES.append(
            (
                "svc",
                C,
                cw,
            )
        )


for C in [
    0.10,
    0.30,
    1.00,
    3.00,
]:

    for cw in [
        "native",
        "balanced",
    ]:

        CANDIDATES.append(
            (
                "logreg",
                C,
                cw,
            )
        )


QWEN_VAL_PATHS = {

    "БАД":
        Path(
            "/kaggle/working/ecup_phase3/"
            "adapters/qwen35_4b_BAD_qlora/"
            "val_predictions.csv"
        ),

    "Легковоспламеняющиеся":
        Path(
            "/kaggle/working/ecup_phase3/"
            "adapters/qwen35_4b_FIRE_qlora/"
            "val_predictions.csv"
        ),
}


def normalize_id(x):

    if (
        isinstance(
            x,
            (float, np.floating),
        )
        and float(x).is_integer()
    ):

        return str(
            int(x)
        )

    s = str(x)

    if (
        s.endswith(".0")
        and
        s[:-2].isdigit()
    ):

        return s[:-2]

    return s


def train_embedding_expert(
    category,
):

    print("\n")
    print("=" * 70)
    print(
        "EMBEDDING EXPERT:",
        category
    )
    print("=" * 70)


    category_mask = (
        df["category"]
        .astype(str)
        .values
        ==
        category
    )


    train_mask = (
        category_mask
        &
        (
            df["split"]
            .astype(str)
            .values
            ==
            "train"
        )
    )


    val_mask = (
        category_mask
        &
        (
            df["split"]
            .astype(str)
            .values
            ==
            "val"
        )
    )


    train_indices = np.flatnonzero(
        train_mask
    )

    val_indices = np.flatnonzero(
        val_mask
    )


    X_train = X_ALL[
        train_indices
    ]

    y_train = (
        df.iloc[
            train_indices
        ]["label"]
        .astype(int)
        .values
    )


    X_val = X_ALL[
        val_indices
    ]

    y_val = (
        df.iloc[
            val_indices
        ]["label"]
        .astype(int)
        .values
    )


    print(
        "TRAIN:",
        len(y_train),
        "| VAL:",
        len(y_val),
    )

    print(
        "Train labels:",
        dict(
            zip(
                *np.unique(
                    y_train,
                    return_counts=True,
                )
            )
        )
    )

    print(
        "Val labels:",
        dict(
            zip(
                *np.unique(
                    y_val,
                    return_counts=True,
                )
            )
        )
    )


    # --------------------------------------------------------
    # 5-fold OOF ONLY ON TRAIN
    # --------------------------------------------------------

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED,
    )


    candidate_rows = []

    best = None


    for family, C, cw in CANDIDATES:

        name = (
            f"{family}_"
            f"C{C}_"
            f"{cw}"
        )


        oof = np.zeros(
            len(y_train),
            dtype=np.float64,
        )


        for fold, (
            fit_idx,
            hold_idx,
        ) in enumerate(
            skf.split(
                X_train,
                y_train,
            )
        ):

            model = make_model(
                family,
                C,
                cw,
            )


            model.fit(
                X_train[
                    fit_idx
                ],
                y_train[
                    fit_idx
                ],
            )


            oof[
                hold_idx
            ] = (
                model
                .decision_function(
                    X_train[
                        hold_idx
                    ]
                )
            )


        default_pred = (
            oof >= 0.0
        ).astype(
            np.int8
        )


        default_f1 = f1_score(
            y_train,
            default_pred,
            zero_division=0,
        )


        threshold, tuned_f1 = (
            best_threshold_from_scores(
                y_train,
                oof,
            )
        )


        row = {

            "model": name,

            "family": family,

            "C": C,

            "class_weight": cw,

            "oof_f1_default":
                float(
                    default_f1
                ),

            "oof_best_threshold":
                float(
                    threshold
                ),

            "oof_f1_tuned":
                float(
                    tuned_f1
                ),
        }


        candidate_rows.append(
            row
        )


        print(
            f"{name:30s} "
            f"OOF default="
            f"{default_f1:.5f} "
            f"tuned="
            f"{tuned_f1:.5f} "
            f"t="
            f"{threshold:.4f}"
        )


        score_key = (
            tuned_f1,
            default_f1,
        )


        if (
            best is None
            or
            score_key
            >
            best["score_key"]
        ):

            best = {
                "family": family,
                "C": C,
                "class_weight": cw,
                "threshold": threshold,
                "oof_score": tuned_f1,
                "score_key": score_key,
            }


    candidate_df = (
        pd.DataFrame(
            candidate_rows
        )
        .sort_values(
            [
                "oof_f1_tuned",
                "oof_f1_default",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )


    print(
        "\nTOP OOF MODELS:"
    )

    display(
        candidate_df.head(10)
    )


    # --------------------------------------------------------
    # Fit selected model on ALL TRAIN.
    #
    # Fixed validation is still untouched during selection.
    # --------------------------------------------------------

    final_model = make_model(
        best["family"],
        best["C"],
        best["class_weight"],
    )


    final_model.fit(
        X_train,
        y_train,
    )


    val_score = (
        final_model
        .decision_function(
            X_val
        )
    )


    val_pred_default = (
        val_score
        >= 0.0
    ).astype(
        np.int8
    )


    val_pred_oof = (
        val_score
        >= best["threshold"]
    ).astype(
        np.int8
    )


    f1_default = f1_score(
        y_val,
        val_pred_default,
        zero_division=0,
    )


    f1_oof = f1_score(
        y_val,
        val_pred_oof,
        zero_division=0,
    )


    print("\n" + "-" * 70)

    print(
        "UNTOUCHED FIXED VALIDATION"
    )

    print("-" * 70)

    print(
        "Selected:",
        best,
    )

    print(
        "VAL F1 default:",
        round(
            float(
                f1_default
            ),
            6,
        )
    )

    print(
        "VAL F1 OOF threshold:",
        round(
            float(
                f1_oof
            ),
            6,
        )
    )


    print(
        "CM default:",
        confusion_matrix(
            y_val,
            val_pred_default,
            labels=[
                0,
                1,
            ],
        ).tolist()
    )

    print(
        "CM OOF threshold:",
        confusion_matrix(
            y_val,
            val_pred_oof,
            labels=[
                0,
                1,
            ],
        ).tolist()
    )


    # --------------------------------------------------------
    # Save validation predictions
    # --------------------------------------------------------

    val_frame = (
        df.iloc[
            val_indices
        ][
            [
                "id",
                "category",
                "label",
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    val_frame[
        "embedding_score"
    ] = val_score


    val_frame[
        "embedding_pred_default"
    ] = val_pred_default


    val_frame[
        "embedding_pred_oof"
    ] = val_pred_oof


    # --------------------------------------------------------
    # Compare with Qwen LoRA
    # --------------------------------------------------------

    qwen_comparison = None

    qwen_path = (
        QWEN_VAL_PATHS[
            category
        ]
    )


    if qwen_path.exists():

        qwen = pd.read_csv(
            qwen_path
        )


        val_frame["_id"] = (
            val_frame["id"]
            .map(
                normalize_id
            )
        )


        qwen["_id"] = (
            qwen["id"]
            .map(
                normalize_id
            )
        )


        if "pred_05" in qwen.columns:

            qwen[
                "qwen_pred"
            ] = (
                qwen[
                    "pred_05"
                ]
                .astype(int)
            )

        elif "p1" in qwen.columns:

            qwen[
                "qwen_pred"
            ] = (
                qwen[
                    "p1"
                ]
                >= 0.5
            ).astype(int)

        else:

            raise ValueError(
                "Qwen val file lacks "
                "pred_05 / p1"
            )


        columns = [
            "_id",
            "qwen_pred",
        ]


        if "p1" in qwen.columns:

            columns.append(
                "p1"
            )


        comparison = (
            val_frame
            .merge(
                qwen[
                    columns
                ],
                on="_id",
                how="inner",
                validate="one_to_one",
            )
        )


        assert (
            len(comparison)
            ==
            len(val_frame)
        ), (
            len(comparison),
            len(val_frame),
        )


        comparison[
            "qwen_correct"
        ] = (
            comparison[
                "qwen_pred"
            ]
            ==
            comparison[
                "label"
            ]
        )


        comparison[
            "embed_correct"
        ] = (
            comparison[
                "embedding_pred_oof"
            ]
            ==
            comparison[
                "label"
            ]
        )


        qwen_f1 = f1_score(
            comparison["label"],
            comparison["qwen_pred"],
            zero_division=0,
        )


        qwen_errors = int(
            (
                ~comparison[
                    "qwen_correct"
                ]
            ).sum()
        )


        embed_errors = int(
            (
                ~comparison[
                    "embed_correct"
                ]
            ).sum()
        )


        both_wrong = int(
            (
                ~comparison[
                    "qwen_correct"
                ]
                &
                ~comparison[
                    "embed_correct"
                ]
            ).sum()
        )


        recoverable = int(
            (
                ~comparison[
                    "qwen_correct"
                ]
                &
                comparison[
                    "embed_correct"
                ]
            ).sum()
        )


        dangerous = int(
            (
                comparison[
                    "qwen_correct"
                ]
                &
                ~comparison[
                    "embed_correct"
                ]
            ).sum()
        )


        print(
            "\n--- QWEN vs EMBEDDING ---"
        )

        print(
            "Qwen F1:",
            round(
                float(
                    qwen_f1
                ),
                6,
            )
        )

        print(
            "Embedding F1:",
            round(
                float(
                    f1_oof
                ),
                6,
            )
        )

        print()

        print(
            "Qwen errors:",
            qwen_errors,
        )

        print(
            "Embedding errors:",
            embed_errors,
        )

        print(
            "Both wrong:",
            both_wrong,
        )

        print(
            "Qwen wrong / Embed correct:",
            recoverable,
            "<-- VERY IMPORTANT",
        )

        print(
            "Qwen correct / Embed wrong:",
            dangerous,
        )


        qwen_comparison = {

            "qwen_f1":
                float(
                    qwen_f1
                ),

            "qwen_errors":
                qwen_errors,

            "embedding_errors":
                embed_errors,

            "both_wrong":
                both_wrong,

            "qwen_wrong_embedding_correct":
                recoverable,

            "qwen_correct_embedding_wrong":
                dangerous,
        }


        comparison.drop(
            columns=[
                "_id"
            ],
            errors="ignore",
        ).to_csv(
            PHASE5
            /
            (
                "BAD"
                if category
                == "БАД"
                else "FIRE"
            )
            /
            "qwen_embedding_comparison.csv",
            index=False,
        )


    # --------------------------------------------------------
    # Save expert
    # --------------------------------------------------------

    category_dir = (
        PHASE5
        /
        (
            "BAD"
            if category
            == "БАД"
            else "FIRE"
        )
    )

    category_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    joblib.dump(
        final_model,
        category_dir
        / "embedding_classifier.joblib",
        compress=3,
    )


    candidate_df.to_csv(
        category_dir
        / "oof_candidates.csv",
        index=False,
    )


    val_frame.drop(
        columns=[
            "_id"
        ],
        errors="ignore",
    ).to_csv(
        category_dir
        / "val_predictions.csv",
        index=False,
    )


    summary = {

        "category":
            category,

        "train_n":
            int(
                len(y_train)
            ),

        "val_n":
            int(
                len(y_val)
            ),

        "embedding_dim":
            int(
                EMBED_DIM
            ),

        "selected_by":
            "5-fold OOF on train only",

        "model_family":
            best["family"],

        "C":
            float(
                best["C"]
            ),

        "class_weight":
            best[
                "class_weight"
            ],

        "oof_f1":
            float(
                best[
                    "oof_score"
                ]
            ),

        "oof_threshold":
            float(
                best[
                    "threshold"
                ]
            ),

        "fixed_val_f1_default":
            float(
                f1_default
            ),

        "fixed_val_f1_oof_threshold":
            float(
                f1_oof
            ),

        "fixed_val_accuracy":
            float(
                accuracy_score(
                    y_val,
                    val_pred_oof,
                )
            ),

        "fixed_val_confusion_matrix":
            confusion_matrix(
                y_val,
                val_pred_oof,
                labels=[
                    0,
                    1,
                ],
            ).tolist(),

        "qwen_comparison":
            qwen_comparison,
    }


    (
        category_dir
        / "summary.json"
    ).write_text(
        json.dumps(
            summary,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )


    print(
        "\nSUMMARY:"
    )

    print(
        json.dumps(
            summary,
            ensure_ascii=False,
            indent=2,
        )
    )


    return summary


# ============================================================
# 11. RUN BOTH CATEGORY CLASSIFIERS
# ============================================================

bad_summary = train_embedding_expert(
    "БАД"
)

fire_summary = train_embedding_expert(
    "Легковоспламеняющиеся"
)


# ============================================================
# 12. FINAL PHASE TABLE
# ============================================================

result_table = pd.DataFrame(
    [
        {
            "category":
                "БАД",

            "OOF train F1":
                bad_summary[
                    "oof_f1"
                ],

            "fixed val F1 default":
                bad_summary[
                    "fixed_val_f1_default"
                ],

            "fixed val F1 OOF threshold":
                bad_summary[
                    "fixed_val_f1_oof_threshold"
                ],

            "Qwen wrong / Embed correct":
                (
                    bad_summary
                    .get(
                        "qwen_comparison"
                    )
                    or {}
                )
                .get(
                    "qwen_wrong_embedding_correct"
                ),
        },

        {
            "category":
                "Легковоспламеняющиеся",

            "OOF train F1":
                fire_summary[
                    "oof_f1"
                ],

            "fixed val F1 default":
                fire_summary[
                    "fixed_val_f1_default"
                ],

            "fixed val F1 OOF threshold":
                fire_summary[
                    "fixed_val_f1_oof_threshold"
                ],

            "Qwen wrong / Embed correct":
                (
                    fire_summary
                    .get(
                        "qwen_comparison"
                    )
                    or {}
                )
                .get(
                    "qwen_wrong_embedding_correct"
                ),
        },
    ]
)


print("\n")
print("=" * 70)
print("PHASE 5 RESULT")
print("=" * 70)

display(
    result_table
)


print(
    "Mean fixed val F1 default:",
    float(
        result_table[
            "fixed val F1 default"
        ].mean()
    )
)

print(
    "Mean fixed val F1 OOF threshold:",
    float(
        result_table[
            "fixed val F1 OOF threshold"
        ].mean()
    )
)


PHASE_RESULT = {

    "model":
        MODEL_ID,

    "input":
        "name + description + all original images",

    "max_pixels_per_image":
        261120,

    "embedding_dim":
        int(
            EMBED_DIM
        ),

    "selection":
        "5-fold OOF on Phase-3 train only",

    "BAD":
        bad_summary,

    "FIRE":
        fire_summary,

    "mean_fixed_val_f1_default":
        float(
            result_table[
                "fixed val F1 default"
            ].mean()
        ),

    "mean_fixed_val_f1_oof_threshold":
        float(
            result_table[
                "fixed val F1 OOF threshold"
            ].mean()
        ),
}


(
    PHASE5
    / "phase5_result.json"
).write_text(
    json.dumps(
        PHASE_RESULT,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 13. ZIP IMPORTANT ARTIFACTS
#
# Do NOT include 2B base model or repo.
# ============================================================

DOWNLOAD_DIR = (
    PHASE5
    / "download"
)

shutil.rmtree(
    DOWNLOAD_DIR,
    ignore_errors=True,
)

DOWNLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Full embeddings are extremely valuable later for
# retrieval / stacking, so save them.
shutil.copy2(
    EMBED_PATH,
    DOWNLOAD_DIR
    / EMBED_PATH.name,
)

shutil.copy2(
    INDEX_PATH,
    DOWNLOAD_DIR
    / INDEX_PATH.name,
)

shutil.copy2(
    PHASE5
    / "phase5_result.json",
    DOWNLOAD_DIR
    / "phase5_result.json",
)


for category_folder in [
    "BAD",
    "FIRE",
]:

    src = (
        PHASE5
        / category_folder
    )

    dst = (
        DOWNLOAD_DIR
        / category_folder
    )

    dst.mkdir(
        parents=True,
        exist_ok=True,
    )

    for filename in [
        "embedding_classifier.joblib",
        "oof_candidates.csv",
        "val_predictions.csv",
        "qwen_embedding_comparison.csv",
        "summary.json",
    ]:

        p = src / filename

        if p.exists():

            shutil.copy2(
                p,
                dst / filename,
            )


ZIP_PATH = Path(
    shutil.make_archive(
        "/kaggle/working/ecup_phase5_embedding",
        "zip",
        root_dir=str(
            DOWNLOAD_DIR
        ),
    )
)


print("\n" + "=" * 70)
print("PHASE 5 FINISHED")
print("=" * 70)

print(
    "ZIP:",
    ZIP_PATH
)

print(
    "SIZE:",
    round(
        ZIP_PATH.stat().st_size
        / 1024**2,
        2,
    ),
    "MB",
)

display(
    FileLink(
        str(
            ZIP_PATH
        )
    )
)

Manifest: /kaggle/working/ecup_phase3/train_manifest.csv
Installing qwen-vl-utils...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 56.5 MB/s eta 0:00:00
qwen-vl-utils installed
Cloning official Qwen3-VL-Embedding repo...


Cloning into '/kaggle/working/Qwen3-VL-Embedding'...


Official embedder: /kaggle/working/Qwen3-VL-Embedding/src/models/qwen3_vl_embedding.py



Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

...

Loading weights: 100%|██████████| 625/625 [00:04<00:00, 130.24it/s]


Model loaded in 8.4 sec
Model loaded in 8.6 sec
chunk 1/102 | rows 0:64 | 0.857 rows/s | ETA ~124.9m
chunk 1/102 | rows 0:64 | 0.849 rows/s | ETA ~126.1m
chunk 2/102 | rows 64:128 | 0.822 rows/s | ETA ~128.9m
chunk 2/102 | rows 64:128 | 0.823 rows/s | ETA ~128.8m
chunk 3/102 | rows 128:192 | 0.812 rows/s | ETA ~129.2m
chunk 3/102 | rows 128:192 | 0.785 rows/s | ETA ~133.6m
chunk 4/102 | rows 192:256 | 0.809 rows/s | ETA ~128.4m
chunk 4/102 | rows 192:256 | 0.786 rows/s | ETA ~132.1m
chunk 5/102 | rows 256:320 | 0.811 rows/s | ETA ~126.7m
chunk 5/102 | rows 256:320 | 0.769 rows/s | ETA ~133.6m
chunk 6/102 | rows 320:384 | 0.813 rows/s | ETA ~125.1m
chunk 6/102 | rows 320:384 | 0.765 rows/s | ETA ~132.9m
chunk 7/102 | rows 384:448 | 0.810 rows/s | ETA ~124.3m
chunk 7/102 | rows 384:448 | 0.763 rows/s | ETA ~132.0m
chunk 8/102 | rows 448:512 | 0.802 rows/s | ETA ~124.2m
chunk 8/102 | rows 448:512 | 0.764 rows/s | ETA ~130.3m
chunk 9/102 | rows 512:576 | 0.805 rows/s | ETA ~122.3m
chunk 9/

,model,family,C,class_weight,oof_f1_default,oof_best_threshold,oof_f1_tuned
0,svc_C3.0_native,svc,3.0,native,0.942368,-0.056086,0.943727
1,svc_C3.0_balanced,svc,3.0,balanced,0.927629,-0.364646,0.942559
2,svc_C1.0_native,svc,1.0,native,0.938813,0.030823,0.940061
3,svc_C1.0_balanced,svc,1.0,balanced,0.924083,-0.437959,0.938693
4,svc_C0.3_native,svc,0.3,native,0.932007,-0.079810,0.933782
5,svc_C0.3_balanced,svc,0.3,balanced,0.914730,-0.370634,0.932819
6,logreg_C3.0_native,logreg,3.0,native,0.930987,-0.127803,0.932590
7,logreg_C3.0_balanced,logreg,3.0,balanced,0.914151,-0.868480,0.931580
8,svc_C0.1_native,svc,0.1,native,0.926549,0.041439,0.927382
9,svc_C0.1_balanced,svc,0.1,balanced,0.905098,-0.336961,0.927371



----------------------------------------------------------------------
UNTOUCHED FIXED VALIDATION
----------------------------------------------------------------------
Selected: {'family': 'svc', 'C': 3.0, 'class_weight': 'native', 'threshold': -0.05608631863904501, 'oof_score': 0.9437269372693727, 'score_key': (0.9437269372693727, 0.9423684699758319)}
VAL F1 default: 0.934866
VAL F1 OOF threshold: 0.933078
CM default: [[68, 20], [14, 244]]
CM OOF threshold: [[67, 21], [14, 244]]

--- QWEN vs EMBEDDING ---
Qwen F1: 0.938856
Embedding F1: 0.933078

Qwen errors: 31
Embedding errors: 35
Both wrong: 13
Qwen wrong / Embed correct: 18 <-- VERY IMPORTANT
Qwen correct / Embed wrong: 22


OSError: Cannot save file into a non-existent directory: '/kaggle/working/ecup_phase5_embedding/BAD'

In [24]:
# ============================================================
# FIX PHASE 5 SAVE DIRECTORIES + CONTINUE
# EMBEDDINGS ARE ALREADY DONE — NO GPU RECOMPUTE
# ============================================================

from pathlib import Path

PHASE5 = Path(
    "/kaggle/working/ecup_phase5_embedding"
)

# Bug fix: these directories must exist BEFORE comparison.to_csv()
(PHASE5 / "BAD").mkdir(
    parents=True,
    exist_ok=True,
)

(PHASE5 / "FIRE").mkdir(
    parents=True,
    exist_ok=True,
)

print("Directories fixed:")
print(PHASE5 / "BAD")
print(PHASE5 / "FIRE")

Directories fixed:
/kaggle/working/ecup_phase5_embedding/BAD
/kaggle/working/ecup_phase5_embedding/FIRE


In [ ]:
bad_summary = train_embedding_expert(
    "БАД"
)

fire_summary = train_embedding_expert(
    "Легковоспламеняющиеся"
)


# ============================================================
# 12. FINAL PHASE TABLE
# ============================================================

result_table = pd.DataFrame(
    [
        {
            "category":
                "БАД",

            "OOF train F1":
                bad_summary[
                    "oof_f1"
                ],

            "fixed val F1 default":
                bad_summary[
                    "fixed_val_f1_default"
                ],

            "fixed val F1 OOF threshold":
                bad_summary[
                    "fixed_val_f1_oof_threshold"
                ],

            "Qwen wrong / Embed correct":
                (
                    bad_summary
                    .get(
                        "qwen_comparison"
                    )
                    or {}
                )
                .get(
                    "qwen_wrong_embedding_correct"
                ),
        },

        {
            "category":
                "Легковоспламеняющиеся",

            "OOF train F1":
                fire_summary[
                    "oof_f1"
                ],

            "fixed val F1 default":
                fire_summary[
                    "fixed_val_f1_default"
                ],

            "fixed val F1 OOF threshold":
                fire_summary[
                    "fixed_val_f1_oof_threshold"
                ],

            "Qwen wrong / Embed correct":
                (
                    fire_summary
                    .get(
                        "qwen_comparison"
                    )
                    or {}
                )
                .get(
                    "qwen_wrong_embedding_correct"
                ),
        },
    ]
)


print("\n")
print("=" * 70)
print("PHASE 5 RESULT")
print("=" * 70)

display(
    result_table
)


print(
    "Mean fixed val F1 default:",
    float(
        result_table[
            "fixed val F1 default"
        ].mean()
    )
)

print(
    "Mean fixed val F1 OOF threshold:",
    float(
        result_table[
            "fixed val F1 OOF threshold"
        ].mean()
    )
)


PHASE_RESULT = {

    "model":
        MODEL_ID,

    "input":
        "name + description + all original images",

    "max_pixels_per_image":
        261120,

    "embedding_dim":
        int(
            EMBED_DIM
        ),

    "selection":
        "5-fold OOF on Phase-3 train only",

    "BAD":
        bad_summary,

    "FIRE":
        fire_summary,

    "mean_fixed_val_f1_default":
        float(
            result_table[
                "fixed val F1 default"
            ].mean()
        ),

    "mean_fixed_val_f1_oof_threshold":
        float(
            result_table[
                "fixed val F1 OOF threshold"
            ].mean()
        ),
}


(
    PHASE5
    / "phase5_result.json"
).write_text(
    json.dumps(
        PHASE_RESULT,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 13. ZIP IMPORTANT ARTIFACTS
#
# Do NOT include 2B base model or repo.
# ============================================================

DOWNLOAD_DIR = (
    PHASE5
    / "download"
)

shutil.rmtree(
    DOWNLOAD_DIR,
    ignore_errors=True,
)

DOWNLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Full embeddings are extremely valuable later for
# retrieval / stacking, so save them.
shutil.copy2(
    EMBED_PATH,
    DOWNLOAD_DIR
    / EMBED_PATH.name,
)

shutil.copy2(
    INDEX_PATH,
    DOWNLOAD_DIR
    / INDEX_PATH.name,
)

shutil.copy2(
    PHASE5
    / "phase5_result.json",
    DOWNLOAD_DIR
    / "phase5_result.json",
)


for category_folder in [
    "BAD",
    "FIRE",
]:

    src = (
        PHASE5
        / category_folder
    )

    dst = (
        DOWNLOAD_DIR
        / category_folder
    )

    dst.mkdir(
        parents=True,
        exist_ok=True,
    )

    for filename in [
        "embedding_classifier.joblib",
        "oof_candidates.csv",
        "val_predictions.csv",
        "qwen_embedding_comparison.csv",
        "summary.json",
    ]:

        p = src / filename

        if p.exists():

            shutil.copy2(
                p,
                dst / filename,
            )


ZIP_PATH = Path(
    shutil.make_archive(
        "/kaggle/working/ecup_phase5_embedding",
        "zip",
        root_dir=str(
            DOWNLOAD_DIR
        ),
    )
)


print("\n" + "=" * 70)
print("PHASE 5 FINISHED")
print("=" * 70)

print(
    "ZIP:",
    ZIP_PATH
)

print(
    "SIZE:",
    round(
        ZIP_PATH.stat().st_size
        / 1024**2,
        2,
    ),
    "MB",
)

display(
    FileLink(
        str(
            ZIP_PATH
        )
    )
)



EMBEDDING EXPERT: БАД
TRAIN: 7123 | VAL: 346
Train labels: {np.int64(0): np.int64(1817), np.int64(1): np.int64(5306)}
Val labels: {np.int64(0): np.int64(88), np.int64(1): np.int64(258)}
svc_C0.03_native               OOF default=0.91561 tuned=0.91924 t=0.0481
svc_C0.03_balanced             OOF default=0.89345 tuned=0.91776 t=-0.3411
svc_C0.1_native                OOF default=0.92655 tuned=0.92738 t=0.0414
svc_C0.1_balanced              OOF default=0.90510 tuned=0.92737 t=-0.3370
svc_C0.3_native                OOF default=0.93201 tuned=0.93378 t=-0.0798
svc_C0.3_balanced              OOF default=0.91473 tuned=0.93282 t=-0.3706
svc_C1.0_native                OOF default=0.93881 tuned=0.94006 t=0.0308
svc_C1.0_balanced              OOF default=0.92408 tuned=0.93869 t=-0.4380
svc_C3.0_native                OOF default=0.94237 tuned=0.94373 t=-0.0561
svc_C3.0_balanced              OOF default=0.92763 tuned=0.94256 t=-0.3646
logreg_C0.1_native             OOF default=0.89226 tuned=0.90783 

,model,family,C,class_weight,oof_f1_default,oof_best_threshold,oof_f1_tuned
0,svc_C3.0_native,svc,3.0,native,0.942368,-0.056086,0.943727
1,svc_C3.0_balanced,svc,3.0,balanced,0.927629,-0.364646,0.942559
2,svc_C1.0_native,svc,1.0,native,0.938813,0.030823,0.940061
3,svc_C1.0_balanced,svc,1.0,balanced,0.924083,-0.437959,0.938693
4,svc_C0.3_native,svc,0.3,native,0.932007,-0.079810,0.933782
5,svc_C0.3_balanced,svc,0.3,balanced,0.914730,-0.370634,0.932819
6,logreg_C3.0_native,logreg,3.0,native,0.930987,-0.127803,0.932590
7,logreg_C3.0_balanced,logreg,3.0,balanced,0.914151,-0.868480,0.931580
8,svc_C0.1_native,svc,0.1,native,0.926549,0.041439,0.927382
9,svc_C0.1_balanced,svc,0.1,balanced,0.905098,-0.336961,0.927371



----------------------------------------------------------------------
UNTOUCHED FIXED VALIDATION
----------------------------------------------------------------------
Selected: {'family': 'svc', 'C': 3.0, 'class_weight': 'native', 'threshold': -0.05608631863904501, 'oof_score': 0.9437269372693727, 'score_key': (0.9437269372693727, 0.9423684699758319)}
VAL F1 default: 0.934866
VAL F1 OOF threshold: 0.933078
CM default: [[68, 20], [14, 244]]
CM OOF threshold: [[67, 21], [14, 244]]

--- QWEN vs EMBEDDING ---
Qwen F1: 0.938856
Embedding F1: 0.933078

Qwen errors: 31
Embedding errors: 35
Both wrong: 13
Qwen wrong / Embed correct: 18 <-- VERY IMPORTANT
Qwen correct / Embed wrong: 22

SUMMARY:
{
  "category": "БАД",
  "train_n": 7123,
  "val_n": 346,
  "embedding_dim": 2048,
  "selected_by": "5-fold OOF on train only",
  "model_family": "svc",
  "C": 3.0,
  "class_weight": "native",
  "oof_f1": 0.9437269372693727,
  "oof_threshold": -0.05608631863904501,
  "fixed_val_f1_default": 0.9348659

In [1]:
print("adf")

adf
